<a href="https://colab.research.google.com/github/scorchbot/fantasy-football-picker/blob/main/Fantasy_Football_Picker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nflreadpy -q

import pandas as pd
import re
import nflreadpy as nfl

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

print("Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.9 MB/s eta 0:00:00
Setup complete


In [2]:
print("Fantasy Football Model is alive!")

Fantasy Football Model is alive!


In [3]:
import pandas as pd

url = "https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats.csv"

players = pd.read_csv(url)

print(f"Downloaded {len(players):,} player-game records")
print(players.head())

Downloaded 134,470 player-game records
    player_id player_name    player_display_name position position_group  \
0  00-0000003         NaN  Abdul-Karim al-Jabbar       RB             RB   
1  00-0000003         NaN  Abdul-Karim al-Jabbar       RB             RB   
2  00-0000003         NaN  Abdul-Karim al-Jabbar       RB             RB   
3  00-0000003         NaN  Abdul-Karim al-Jabbar       RB             RB   
4  00-0000003         NaN  Abdul-Karim al-Jabbar       RB             RB   

  headshot_url recent_team  season  week season_type  ...  \
0          NaN         MIA    1999     1         REG  ...   
1          NaN         MIA    1999     2         REG  ...   
2          NaN         MIA    1999     4         REG  ...   
3          NaN         CLE    1999     7         REG  ...   
4          NaN         CLE    1999     8         REG  ...   

  receiving_first_downs  receiving_epa  receiving_2pt_conversions  racr  \
0                     0       0.292378                        

In [4]:
players.columns.tolist()

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

In [5]:
# Let's look at the player data we're working with
players[['player_name', 'position', 'season', 'week',
         'rushing_yards', 'rushing_tds',
         'receptions', 'receiving_yards', 'receiving_tds',
         'fantasy_points_ppr']].head(20)

,player_name,position,season,week,rushing_yards,rushing_tds,receptions,receiving_yards,receiving_tds,fantasy_points_ppr
0,NaN,RB,1999,1,60,1,1,7,0,13.70
1,NaN,RB,1999,2,33,0,3,18,0,8.10
2,NaN,RB,1999,4,2,0,0,0,0,0.20
3,NaN,RB,1999,7,27,0,2,8,0,5.50
4,NaN,RB,1999,8,39,0,0,0,0,3.90
5,NaN,RB,1999,9,23,0,1,2,0,3.50
6,NaN,RB,1999,10,54,0,1,7,0,7.10
7,NaN,RB,1999,11,11,0,2,2,0,3.30
8,NaN,RB,1999,12,35,0,1,1,0,4.60
9,NaN,RB,1999,13,29,0,1,21,0,6.00


In [6]:
recent = players[
    (players['season'] >= 2021) &
    (players['season'] <= 2025)
].copy()

print(f"{len(recent):,} player-game records")
print(f"Seasons: {recent['season'].min()}–{recent['season'].max()}")

22,579 player-game records
Seasons: 2021–2024


In [7]:
recent[['player_display_name', 'position', 'season', 'week',
        'recent_team', 'opponent_team',
        'rushing_yards', 'rushing_tds',
        'receptions', 'receiving_yards', 'receiving_tds',
        'fantasy_points_ppr']].head(20)

,player_display_name,position,season,week,recent_team,opponent_team,rushing_yards,rushing_tds,receptions,receiving_yards,receiving_tds,fantasy_points_ppr
111891,Tom Brady,QB,2021,1,TB,DAL,0,0,0,0,0,27.16
111892,Tom Brady,QB,2021,2,TB,ATL,6,0,0,0,0,29.64
111893,Tom Brady,QB,2021,3,TB,LA,14,1,0,0,0,28.68
111894,Tom Brady,QB,2021,4,TB,NE,3,0,0,0,0,11.06
111895,Tom Brady,QB,2021,5,TB,MIA,13,0,0,0,0,37.74
111896,Tom Brady,QB,2021,6,TB,PHI,1,0,0,0,0,17.98
111897,Tom Brady,QB,2021,7,TB,CHI,0,0,0,0,0,24.44
111898,Tom Brady,QB,2021,8,TB,NO,2,0,0,0,0,25.20
111899,Tom Brady,QB,2021,10,TB,WAS,2,0,0,0,0,13.00
111900,Tom Brady,QB,2021,11,TB,NYG,10,0,0,0,0,19.28


In [8]:
def calculate_fantasy_points(row):
    points = 0

    # Passing
    points += row['completions'] * -0.1
    points += row['passing_yards'] / 30
    points += row['passing_tds'] * 5
    points += row['interceptions'] * -2

    # Rushing
    points += row['rushing_yards'] / 10
    points += row['rushing_tds'] * 6

    # Rushing yardage bonuses
    if row['rushing_yards'] >= 300:
        points += 5
    elif row['rushing_yards'] >= 200:
        points += 3
    elif row['rushing_yards'] >= 150:
        points += 2

    # Receiving
    points += row['receptions'] * 0.5
    points += row['receiving_yards'] / 10
    points += row['receiving_tds'] * 6

    # Receiving yardage bonuses
    if row['receiving_yards'] >= 250:
        points += 3
    elif row['receiving_yards'] >= 200:
        points += 2
    elif row['receiving_yards'] >= 150:
        points += 1

    # 2-point conversions
    points += (
        row['passing_2pt_conversions']
        + row['rushing_2pt_conversions']
        + row['receiving_2pt_conversions']
    ) * 2

    # Fumbles lost
    points += (
        row['rushing_fumbles_lost']
        + row['receiving_fumbles_lost']
        + row['sack_fumbles_lost']
    ) * -2

    # Passing yardage bonuses
    if row['passing_yards'] >= 500:
        points += 4
    elif row['passing_yards'] >= 400:
        points += 3
    elif row['passing_yards'] >= 300:
        points += 2

    return round(points, 2)

In [9]:
recent['my_fantasy_points'] = recent.apply(
    calculate_fantasy_points,
    axis=1
)

recent[['player_display_name',
        'position',
        'season',
        'week',
        'fantasy_points_ppr',
        'my_fantasy_points']].head(20)

,player_display_name,position,season,week,fantasy_points_ppr,my_fantasy_points
111891,Tom Brady,QB,2021,1,27.16,27.43
111892,Tom Brady,QB,2021,2,29.64,30.40
111893,Tom Brady,QB,2021,3,28.68,25.70
111894,Tom Brady,QB,2021,4,11.06,7.07
111895,Tom Brady,QB,2021,5,37.74,40.00
111896,Tom Brady,QB,2021,6,17.98,14.60
111897,Tom Brady,QB,2021,7,24.44,25.03
111898,Tom Brady,QB,2021,8,25.20,25.90
111899,Tom Brady,QB,2021,10,13.00,11.23
111900,Tom Brady,QB,2021,11,19.28,18.23


In [10]:
recent[
    (recent['player_display_name'] == 'Tom Brady') &
    (recent['season'] == 2021) &
    (recent['week'] == 1)
][[
    'player_display_name',
    'completions',
    'passing_yards',
    'passing_tds',
    'interceptions',
    'carries',
    'rushing_yards',
    'rushing_tds',
    'receptions',
    'receiving_yards',
    'receiving_tds',
    'passing_2pt_conversions',
    'rushing_2pt_conversions',
    'receiving_2pt_conversions',
    'rushing_fumbles_lost',
    'receiving_fumbles_lost',
    'sack_fumbles_lost',
    'fantasy_points_ppr',
    'my_fantasy_points'
]]

,player_display_name,completions,passing_yards,passing_tds,interceptions,carries,rushing_yards,rushing_tds,receptions,receiving_yards,receiving_tds,passing_2pt_conversions,rushing_2pt_conversions,receiving_2pt_conversions,rushing_fumbles_lost,receiving_fumbles_lost,sack_fumbles_lost,fantasy_points_ppr,my_fantasy_points
111891,Tom Brady,32,379,4,2,0,0,0,0,0,0,0,0,0,0,0,0,27.16,27.43


In [11]:
difference = recent['my_fantasy_points'] - recent['fantasy_points_ppr']

print("Average difference:", round(difference.mean(), 2))
print("Largest difference:", round(difference.max(), 2))
print("Smallest difference:", round(difference.min(), 2))

Average difference: -1.3
Largest difference: 3.06
Smallest difference: -12.0


In [12]:
recent[['player_display_name', 'position', 'season', 'week',
        'fantasy_points_ppr', 'my_fantasy_points']].sort_values(
    'my_fantasy_points',
    ascending=False
).head(20)

,player_display_name,position,season,week,fantasy_points_ppr,my_fantasy_points
119412,Joe Mixon,RB,2022,9,55.10,55.10
116289,Jonathan Taylor,RB,2021,11,53.40,53.90
117288,Ja'Marr Chase,WR,2021,17,55.60,53.10
131787,Ja'Marr Chase,WR,2024,10,55.40,52.90
130456,Josh Allen,QB,2024,14,51.88,52.40
128604,De'Von Achane,RB,2023,3,51.30,52.30
116244,Gabe Davis,WR,2021,20,52.10,50.10
123663,Amari Cooper,WR,2023,16,51.50,49.00
120934,Josh Jacobs,RB,2022,12,48.30,48.30
130410,Saquon Barkley,RB,2024,12,46.20,47.20


In [13]:
schedule_url = "https://github.com/nflverse/nflverse-data/releases/download/schedules/games.csv"

schedule = pd.read_csv(schedule_url)

print(f"Downloaded {len(schedule):,} games")
print(schedule.columns.tolist())

Downloaded 7,548 games
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [14]:
# Let's look at the schedule for Tom Brady's Week 1, 2021 game
schedule[
    (schedule['season'] == 2021) &
    (schedule['week'] == 1) &
    (
        (schedule['home_team'] == 'TB') |
        (schedule['away_team'] == 'TB')
    )
][[
    'gameday',
    'away_team',
    'home_team',
    'spread_line',
    'total_line',
    'roof',
    'temp',
    'wind',
    'away_rest',
    'home_rest',
    'away_qb_name',
    'home_qb_name'
]]

,gameday,away_team,home_team,spread_line,total_line,roof,temp,wind,away_rest,home_rest,away_qb_name,home_qb_name
5852,2021-09-09,DAL,TB,10.0,52.5,outdoors,82.0,9.0,7,7,Dak Prescott,Tom Brady


In [15]:
# Build one row per team per game

home_games = schedule[[
    'game_id',
    'season',
    'week',
    'gameday',
    'home_team',
    'away_team',
    'spread_line',
    'total_line',
    'roof',
    'temp',
    'wind',
    'home_rest',
    'home_qb_name'
]].copy()

home_games = home_games.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_rest': 'rest_days',
    'home_qb_name': 'starting_qb'
})

home_games['is_home'] = True
home_games['team_spread'] = -home_games['spread_line']


away_games = schedule[[
    'game_id',
    'season',
    'week',
    'gameday',
    'away_team',
    'home_team',
    'spread_line',
    'total_line',
    'roof',
    'temp',
    'wind',
    'away_rest',
    'away_qb_name'
]].copy()

away_games = away_games.rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_rest': 'rest_days',
    'away_qb_name': 'starting_qb'
})

away_games['is_home'] = False
away_games['team_spread'] = away_games['spread_line']


team_games = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

print(f"Created {len(team_games):,} team-game records")

Created 15,096 team-game records


In [16]:
team_games[
    (team_games['season'] == 2021) &
    (team_games['week'] == 1) &
    (team_games['team'].isin(['TB', 'DAL']))
][[
    'team',
    'opponent',
    'is_home',
    'team_spread',
    'total_line',
    'roof',
    'temp',
    'wind',
    'rest_days',
    'starting_qb'
]]

,team,opponent,is_home,team_spread,total_line,roof,temp,wind,rest_days,starting_qb
5852,TB,DAL,True,-10.0,52.5,outdoors,82.0,9.0,7,Tom Brady
13400,DAL,TB,False,10.0,52.5,outdoors,82.0,9.0,7,Dak Prescott


In [17]:
team_games[
    (team_games['season'] == 2021) &
    (team_games['week'] == 1) &
    (team_games['team'].isin(['TB', 'DAL']))
][[
    'team',
    'opponent',
    'is_home',
    'team_spread',
    'total_line',
    'roof',
    'temp',
    'wind',
    'rest_days',
    'starting_qb'
]]

,team,opponent,is_home,team_spread,total_line,roof,temp,wind,rest_days,starting_qb
5852,TB,DAL,True,-10.0,52.5,outdoors,82.0,9.0,7,Tom Brady
13400,DAL,TB,False,10.0,52.5,outdoors,82.0,9.0,7,Dak Prescott


In [18]:
# Create a schedule row for each team in each game

home_games = schedule[[
    'game_id',
    'season',
    'week',
    'gameday',
    'home_team',
    'away_team',
    'spread_line',
    'total_line',
    'roof',
    'temp',
    'wind',
    'home_rest',
    'home_qb_name'
]].copy()

home_games = home_games.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_rest': 'rest_days',
    'home_qb_name': 'starting_qb'
})

home_games['is_home'] = True
home_games['team_spread'] = -home_games['spread_line']

# Away-team version
away_games = schedule[[
    'game_id',
    'season',
    'week',
    'gameday',
    'away_team',
    'home_team',
    'spread_line',
    'total_line',
    'roof',
    'temp',
    'wind',
    'away_rest',
    'away_qb_name'
]].copy()

away_games = away_games.rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_rest': 'rest_days',
    'away_qb_name': 'starting_qb'
})

away_games['is_home'] = False
away_games['team_spread'] = away_games['spread_line']

# Combine home and away into one team-game table
team_games = pd.concat([home_games, away_games], ignore_index=True)

print(f"Created {len(team_games):,} team-game records")
team_games.head()

Created 15,096 team-game records


,game_id,season,week,gameday,team,opponent,spread_line,total_line,roof,temp,wind,rest_days,starting_qb,is_home,team_spread
0,1999_01_MIN_ATL,1999,1,1999-09-12,ATL,MIN,-4.0,49.0,dome,NaN,NaN,7,Chris Chandler,True,4.0
1,1999_01_KC_CHI,1999,1,1999-09-12,CHI,KC,-3.0,38.0,outdoors,80.0,12.0,7,Shane Matthews,True,3.0
2,1999_01_PIT_CLE,1999,1,1999-09-12,CLE,PIT,-6.0,37.0,outdoors,78.0,12.0,7,Ty Detmer,True,6.0
3,1999_01_OAK_GB,1999,1,1999-09-12,GB,OAK,9.0,43.0,outdoors,67.0,10.0,7,Brett Favre,True,-9.0
4,1999_01_BUF_IND,1999,1,1999-09-12,IND,BUF,-3.0,45.5,dome,NaN,NaN,7,Peyton Manning,True,3.0


In [19]:
model_data = recent.merge(
    team_games,
    left_on=['season', 'week', 'recent_team'],
    right_on=['season', 'week', 'team'],
    how='left'
)

print(f"Player-game rows after join: {len(model_data):,}")
print(f"Rows missing game data: {model_data['game_id'].isna().sum():,}")

Player-game rows after join: 22,579
Rows missing game data: 0


In [20]:
model_data[
    (model_data['player_display_name'] == 'Tom Brady') &
    (model_data['season'] == 2021) &
    (model_data['week'] == 1)
][[
    'player_display_name',
    'position',
    'recent_team',
    'opponent_team',
    'is_home',
    'team_spread',
    'total_line',
    'roof',
    'temp',
    'wind',
    'rest_days',
    'my_fantasy_points'
]]

,player_display_name,position,recent_team,opponent_team,is_home,team_spread,total_line,roof,temp,wind,rest_days,my_fantasy_points
0,Tom Brady,QB,TB,DAL,True,-10.0,52.5,outdoors,82.0,9.0,7,27.43


In [21]:
# Sort so each player's games are in chronological order
model_data = model_data.sort_values(
    ['player_id', 'season', 'week']
).copy()

# 3-game rolling averages using ONLY prior games
for col in [
    'my_fantasy_points',
    'carries',
    'targets',
    'receptions',
    'rushing_yards',
    'receiving_yards',
    'target_share'
]:
    model_data[f'{col}_last3'] = (
        model_data
        .groupby('player_id')[col]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

print("Pregame form features created.")

Pregame form features created.


In [22]:
model_data[
    (model_data['player_display_name'] == 'Jonathan Taylor') &
    (model_data['season'] == 2021)
][[
    'week',
    'my_fantasy_points',
    'my_fantasy_points_last3',
    'carries',
    'carries_last3',
    'targets',
    'targets_last3'
]].head(12)

,week,my_fantasy_points,my_fantasy_points_last3,carries,carries_last3,targets,targets_last3
4388,1,14.6,NaN,17,NaN,7,NaN
4389,2,5.8,14.600000,15,17.000000,1,7.000000
4390,3,7.7,10.200000,10,16.000000,1,4.000000
4391,4,18.9,9.366667,16,14.000000,3,3.000000
4392,5,30.4,10.800000,15,13.666667,4,1.666667
4393,6,28.3,19.000000,14,13.666667,2,2.666667
4394,7,16.5,25.866667,18,15.000000,3,3.000000
4395,8,19.7,25.066667,16,15.666667,4,3.000000
4396,9,35.0,21.500000,19,16.000000,2,3.000000
4397,10,21.6,23.733333,21,17.666667,8,3.000000


In [23]:
# Keep only regular-season games and fantasy-relevant offensive positions
model_data = model_data[
    (model_data['season_type'] == 'REG') &
    (model_data['position'].isin(['QB', 'RB', 'WR', 'TE']))
].copy()

# Put games in chronological order
model_data = model_data.sort_values(
    ['player_id', 'season', 'week']
).copy()

# Rebuild our prior-3-game features,
# restarting the calculation each season
for col in [
    'my_fantasy_points',
    'carries',
    'targets',
    'receptions',
    'rushing_yards',
    'receiving_yards',
    'target_share'
]:
    model_data[f'{col}_last3'] = (
        model_data
        .groupby(['player_id', 'season'])[col]
        .transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    )

print(f"Fantasy-relevant regular-season player-games: {len(model_data):,}")

Fantasy-relevant regular-season player-games: 21,071


In [24]:
model_data[
    (model_data['player_display_name'] == 'Jonathan Taylor') &
    (model_data['season'] == 2021)
][[
    'week',
    'my_fantasy_points',
    'my_fantasy_points_last3',
    'carries',
    'carries_last3',
    'targets',
    'targets_last3'
]].head(12)

,week,my_fantasy_points,my_fantasy_points_last3,carries,carries_last3,targets,targets_last3
4388,1,14.6,NaN,17,NaN,7,NaN
4389,2,5.8,14.600000,15,17.000000,1,7.000000
4390,3,7.7,10.200000,10,16.000000,1,4.000000
4391,4,18.9,9.366667,16,14.000000,3,3.000000
4392,5,30.4,10.800000,15,13.666667,4,1.666667
4393,6,28.3,19.000000,14,13.666667,2,2.666667
4394,7,16.5,25.866667,18,15.000000,3,3.000000
4395,8,19.7,25.066667,16,15.666667,4,3.000000
4396,9,35.0,21.500000,19,16.000000,2,3.000000
4397,10,21.6,23.733333,21,17.666667,8,3.000000


In [25]:
# Build a table of fantasy points allowed by each defense to each position
defense_games = (
    model_data
    .groupby([
        'season',
        'week',
        'opponent_team',
        'position'
    ])['my_fantasy_points']
    .sum()
    .reset_index()
    .rename(columns={
        'opponent_team': 'defense_team',
        'my_fantasy_points': 'fantasy_points_allowed'
    })
)

# Sort in time order
defense_games = defense_games.sort_values(
    ['defense_team', 'position', 'season', 'week']
).copy()

# Prior 3-game average fantasy points allowed
defense_games['defense_fp_allowed_last3'] = (
    defense_games
    .groupby(['defense_team', 'position', 'season'])['fantasy_points_allowed']
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
)

defense_games.head()

,season,week,defense_team,position,fantasy_points_allowed,defense_fp_allowed_last3
0,2021,1,ARI,QB,11.67,NaN
128,2021,2,ARI,QB,24.43,11.670000
256,2021,3,ARI,QB,4.80,18.050000
384,2021,4,ARI,QB,16.83,13.633333
512,2021,5,ARI,QB,11.80,15.353333


In [26]:
model_data = model_data.merge(
    defense_games[[
        'season',
        'week',
        'defense_team',
        'position',
        'defense_fp_allowed_last3'
    ]],
    left_on=[
        'season',
        'week',
        'opponent_team',
        'position'
    ],
    right_on=[
        'season',
        'week',
        'defense_team',
        'position'
    ],
    how='left'
)

print(
    "Rows with defensive matchup data:",
    model_data['defense_fp_allowed_last3'].notna().sum()
)

print(
    "Rows missing defensive matchup data:",
    model_data['defense_fp_allowed_last3'].isna().sum()
)

Rows with defensive matchup data: 19844
Rows missing defensive matchup data: 1227


In [27]:
model_data[
    (model_data['player_display_name'] == 'Jonathan Taylor') &
    (model_data['season'] == 2021)
][[
    'week',
    'opponent_team',
    'my_fantasy_points',
    'my_fantasy_points_last3',
    'defense_fp_allowed_last3'
]].head(12)

,week,opponent_team,my_fantasy_points,my_fantasy_points_last3,defense_fp_allowed_last3
11997,1,SEA,14.6,NaN,NaN
11998,2,LA,5.8,14.600000,24.300000
11999,3,TEN,7.7,10.200000,18.950000
12000,4,MIA,18.9,9.366667,26.433333
12001,5,BAL,30.4,10.800000,19.766667
12002,6,HOU,28.3,19.000000,18.600000
12003,7,SF,16.5,25.866667,17.400000
12004,8,TEN,19.7,25.066667,15.666667
12005,9,NYJ,35.0,21.500000,36.066667
12006,10,JAX,21.6,23.733333,10.133333


In [28]:
# Keep rows where we actually have prior-game information
baseline_data = model_data[
    model_data['my_fantasy_points_last3'].notna()
].copy()

baseline_data['baseline_prediction'] = baseline_data['my_fantasy_points_last3']

baseline_data['baseline_error'] = (
    baseline_data['my_fantasy_points'] -
    baseline_data['baseline_prediction']
).abs()

print(
    "Baseline MAE:",
    round(baseline_data['baseline_error'].mean(), 2)
)

Baseline MAE: 4.86


In [29]:
rb_baseline = baseline_data[
    baseline_data['position'] == 'RB'
].copy()

print(
    "RB baseline MAE:",
    round(rb_baseline['baseline_error'].mean(), 2)
)

print(
    "RB player-games:",
    len(rb_baseline)
)

RB baseline MAE: 5.02
RB player-games: 4868


In [30]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# RBs only for our first experiment
rb_data = model_data[
    (model_data['position'] == 'RB') &
    (model_data['my_fantasy_points_last3'].notna())
].copy()

features = [
    'my_fantasy_points_last3',
    'carries_last3',
    'targets_last3',
    'rushing_yards_last3',
    'receiving_yards_last3',
    'target_share_last3',
    'defense_fp_allowed_last3',
    'team_spread',
    'total_line',
    'is_home',
    'temp',
    'wind',
    'rest_days'
]

# Training data = 2021-2023
train = rb_data[rb_data['season'] <= 2023].copy()

# Test data = 2024
test = rb_data[rb_data['season'] == 2024].copy()

X_train = train[features]
y_train = train['my_fantasy_points']

X_test = test[features]
y_test = test['my_fantasy_points']

# Fill missing values using training-set medians only
medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

model_mae = mean_absolute_error(y_test, predictions)

print(f"Training RB games: {len(train):,}")
print(f"2024 test RB games: {len(test):,}")
print(f"Model MAE: {model_mae:.2f}")

Training RB games: 3,660
2024 test RB games: 1,208
Model MAE: 4.67


In [31]:
baseline_2024_mae = mean_absolute_error(
    y_test,
    test['my_fantasy_points_last3']
)

print(f"2024 baseline MAE: {baseline_2024_mae:.2f}")
print(f"2024 model MAE:    {model_mae:.2f}")

improvement = (
    (baseline_2024_mae - model_mae)
    / baseline_2024_mae
) * 100

print(f"Improvement: {improvement:.1f}%")

2024 baseline MAE: 4.85
2024 model MAE:    4.67
Improvement: 3.8%


In [32]:
import pandas as pd

importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

importance

,feature,importance
1,carries_last3,0.487716
0,my_fantasy_points_last3,0.115044
3,rushing_yards_last3,0.094812
7,team_spread,0.049135
6,defense_fp_allowed_last3,0.043994
2,targets_last3,0.043521
5,target_share_last3,0.043109
4,receiving_yards_last3,0.041859
8,total_line,0.029611
10,temp,0.022746


In [33]:
test = test.copy()
test['model_prediction'] = predictions

In [34]:
from scipy.stats import spearmanr

baseline_rank = spearmanr(
    test['my_fantasy_points_last3'],
    test['my_fantasy_points']
).statistic

model_rank = spearmanr(
    test['model_prediction'],
    test['my_fantasy_points']
).statistic

print(f"Baseline ranking correlation: {baseline_rank:.3f}")
print(f"Model ranking correlation:    {model_rank:.3f}")

Baseline ranking correlation: 0.585
Model ranking correlation:    0.590


In [35]:
feature_groups = {
    '1. Recent performance': [
        'my_fantasy_points_last3'
    ],

    '2. + Workload': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3'
    ],

    '3. + Opponent': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3'
    ],

    '4. + Vegas': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line'
    ],

    '5. + Location/rest': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'is_home',
        'rest_days'
    ],

    '6. + Weather': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'is_home',
        'rest_days',
        'temp',
        'wind'
    ]
}

results = []

for name, cols in feature_groups.items():

    X_train_group = train[cols].copy()
    X_test_group = test[cols].copy()

    medians = X_train_group.median(numeric_only=True)

    X_train_group = X_train_group.fillna(medians)
    X_test_group = X_test_group.fillna(medians)

    m = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    m.fit(X_train_group, y_train)

    preds = m.predict(X_test_group)

    mae = mean_absolute_error(y_test, preds)

    rank = spearmanr(
        preds,
        y_test
    ).statistic

    results.append({
        'Model': name,
        'MAE': round(mae, 3),
        'Rank correlation': round(rank, 3)
    })

pd.DataFrame(results)

,Model,MAE,Rank correlation
0,1. Recent performance,4.818,0.570
1,2. + Workload,4.698,0.591
2,3. + Opponent,4.695,0.591
3,4. + Vegas,4.659,0.591
4,5. + Location/rest,4.661,0.589
5,6. + Weather,4.668,0.590


In [36]:
# Make sure we're in chronological order
model_data = model_data.sort_values(
    ['player_id', 'season', 'week']
).copy()

# Add prior-5-game averages
for col in [
    'my_fantasy_points',
    'carries',
    'targets',
    'receptions',
    'rushing_yards',
    'receiving_yards',
    'target_share'
]:
    model_data[f'{col}_last5'] = (
        model_data
        .groupby(['player_id', 'season'])[col]
        .transform(
            lambda x: x.shift(1).rolling(5, min_periods=1).mean()
        )
    )

# Trend = recent 3-game workload compared with 5-game workload
model_data['carry_trend'] = (
    model_data['carries_last3'] -
    model_data['carries_last5']
)

model_data['target_trend'] = (
    model_data['targets_last3'] -
    model_data['targets_last5']
)

model_data['fantasy_trend'] = (
    model_data['my_fantasy_points_last3'] -
    model_data['my_fantasy_points_last5']
)

print("Longer-term and trend features created.")

Longer-term and trend features created.


In [37]:
rb_data = model_data[
    (model_data['position'] == 'RB') &
    (model_data['my_fantasy_points_last3'].notna())
].copy()

train = rb_data[rb_data['season'] <= 2023].copy()
test = rb_data[rb_data['season'] == 2024].copy()

trend_features = [
    'my_fantasy_points_last3',
    'my_fantasy_points_last5',

    'carries_last3',
    'carries_last5',
    'carry_trend',

    'targets_last3',
    'targets_last5',
    'target_trend',

    'rushing_yards_last3',
    'receiving_yards_last3',
    'target_share_last3',

    'fantasy_trend',

    'defense_fp_allowed_last3',

    'team_spread',
    'total_line'
]

X_train = train[trend_features].copy()
X_test = test[trend_features].copy()

medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

trend_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

trend_model.fit(
    X_train,
    train['my_fantasy_points']
)

trend_predictions = trend_model.predict(X_test)

trend_mae = mean_absolute_error(
    test['my_fantasy_points'],
    trend_predictions
)

trend_rank = spearmanr(
    trend_predictions,
    test['my_fantasy_points']
).statistic

print(f"Previous best MAE: 4.659")
print(f"Trend model MAE:   {trend_mae:.3f}")
print()
print(f"Previous best rank: 0.591")
print(f"Trend model rank:   {trend_rank:.3f}")

Previous best MAE: 4.659
Trend model MAE:   4.659

Previous best rank: 0.591
Trend model rank:   0.590


In [38]:
!pip install nflreadpy -q

In [39]:
import nflreadpy as nfl

print("nflreadpy loaded successfully")

nflreadpy loaded successfully


In [40]:
snap_counts = nfl.load_snap_counts([2021, 2022, 2023, 2024])

print(snap_counts.shape)
print(snap_counts.columns)
snap_counts.head()

(106004, 16)
['game_id', 'pfr_game_id', 'season', 'game_type', 'week', 'player', 'pfr_player_id', 'position', 'team', 'opponent', 'offense_snaps', 'offense_pct', 'defense_snaps', 'defense_pct', 'st_snaps', 'st_pct']


game_id,pfr_game_id,season,game_type,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
str,str,i32,str,i32,str,str,str,str,str,f64,f64,f64,f64,f64,f64
"""2021_01_ARI_TEN""","""202109120oti""",2021,"""REG""",1,"""David Quessenberry""","""QuesDa00""","""G""","""TEN""","""ARI""",64.0,1.0,0.0,0.0,3.0,0.11
"""2021_01_ARI_TEN""","""202109120oti""",2021,"""REG""",1,"""Nate Davis""","""DaviNa01""","""G""","""TEN""","""ARI""",64.0,1.0,0.0,0.0,3.0,0.11
"""2021_01_ARI_TEN""","""202109120oti""",2021,"""REG""",1,"""Rodger Saffold""","""SaffRo20""","""G""","""TEN""","""ARI""",64.0,1.0,0.0,0.0,3.0,0.11
"""2021_01_ARI_TEN""","""202109120oti""",2021,"""REG""",1,"""Ryan Tannehill""","""TannRy00""","""QB""","""TEN""","""ARI""",64.0,1.0,0.0,0.0,0.0,0.0
"""2021_01_ARI_TEN""","""202109120oti""",2021,"""REG""",1,"""Ben Jones""","""JoneBe01""","""C""","""TEN""","""ARI""",64.0,1.0,0.0,0.0,0.0,0.0


In [41]:
snap_pd = snap_counts.to_pandas()

In [42]:
snap_pd = snap_pd[
    snap_pd['position'].isin(['QB', 'RB', 'WR', 'TE'])
].copy()

In [43]:
model_data = model_data.merge(
    snap_pd[[
        'season',
        'week',
        'player',
        'team',
        'opponent',
        'offense_snaps',
        'offense_pct'
    ]],
    left_on=[
        'season',
        'week',
        'player_display_name',
        'recent_team',
        'opponent_team'
    ],
    right_on=[
        'season',
        'week',
        'player',
        'team',
        'opponent'
    ],
    how='left'
)

print("Rows:", len(model_data))
print(
    "Rows with snap data:",
    model_data['offense_pct'].notna().sum()
)
print(
    "Rows missing snap data:",
    model_data['offense_pct'].isna().sum()
)

Rows: 21071
Rows with snap data: 20208
Rows missing snap data: 863


In [44]:
model_data[
    model_data['offense_pct'].notna()
][[
    'player_display_name',
    'position',
    'season',
    'week',
    'offense_snaps',
    'offense_pct',
    'my_fantasy_points'
]].head(20)

,player_display_name,position,season,week,offense_snaps,offense_pct,my_fantasy_points
0,Tom Brady,QB,2021,1,65.0,1.00,27.43
1,Tom Brady,QB,2021,2,60.0,0.95,30.40
2,Tom Brady,QB,2021,3,73.0,1.00,25.70
3,Tom Brady,QB,2021,4,78.0,1.00,7.07
4,Tom Brady,QB,2021,5,62.0,0.84,40.00
5,Tom Brady,QB,2021,6,75.0,1.00,14.60
6,Tom Brady,QB,2021,7,65.0,0.89,25.03
7,Tom Brady,QB,2021,8,61.0,1.00,25.90
8,Tom Brady,QB,2021,10,48.0,1.00,11.23
9,Tom Brady,QB,2021,11,69.0,0.88,18.23


In [45]:
model_data[
    model_data['offense_pct'].notna()
][[
    'player_display_name',
    'position',
    'season',
    'week',
    'offense_snaps',
    'offense_pct',
    'my_fantasy_points'
]].head(20)

,player_display_name,position,season,week,offense_snaps,offense_pct,my_fantasy_points
0,Tom Brady,QB,2021,1,65.0,1.00,27.43
1,Tom Brady,QB,2021,2,60.0,0.95,30.40
2,Tom Brady,QB,2021,3,73.0,1.00,25.70
3,Tom Brady,QB,2021,4,78.0,1.00,7.07
4,Tom Brady,QB,2021,5,62.0,0.84,40.00
5,Tom Brady,QB,2021,6,75.0,1.00,14.60
6,Tom Brady,QB,2021,7,65.0,0.89,25.03
7,Tom Brady,QB,2021,8,61.0,1.00,25.90
8,Tom Brady,QB,2021,10,48.0,1.00,11.23
9,Tom Brady,QB,2021,11,69.0,0.88,18.23


In [46]:
model_data[
    model_data['offense_pct'].notna()
][[
    'player_display_name',
    'position',
    'season',
    'week',
    'offense_snaps',
    'offense_pct',
    'my_fantasy_points'
]].head(20)

,player_display_name,position,season,week,offense_snaps,offense_pct,my_fantasy_points
0,Tom Brady,QB,2021,1,65.0,1.00,27.43
1,Tom Brady,QB,2021,2,60.0,0.95,30.40
2,Tom Brady,QB,2021,3,73.0,1.00,25.70
3,Tom Brady,QB,2021,4,78.0,1.00,7.07
4,Tom Brady,QB,2021,5,62.0,0.84,40.00
5,Tom Brady,QB,2021,6,75.0,1.00,14.60
6,Tom Brady,QB,2021,7,65.0,0.89,25.03
7,Tom Brady,QB,2021,8,61.0,1.00,25.90
8,Tom Brady,QB,2021,10,48.0,1.00,11.23
9,Tom Brady,QB,2021,11,69.0,0.88,18.23


In [47]:
model_data[
    model_data['offense_pct'].isna()
][[
    'player_display_name',
    'position',
    'season',
    'week',
    'recent_team',
    'opponent_team'
]].head(30)

,player_display_name,position,season,week,recent_team,opponent_team
1548,Odell Beckham,WR,2021,3,CLE,CHI
1549,Odell Beckham,WR,2021,4,CLE,MIN
1550,Odell Beckham,WR,2021,5,CLE,LAC
1551,Odell Beckham,WR,2021,6,CLE,ARI
1552,Odell Beckham,WR,2021,7,CLE,DEN
1553,Odell Beckham,WR,2021,8,CLE,PIT
1554,Odell Beckham,WR,2021,10,LA,SF
1555,Odell Beckham,WR,2021,12,LA,GB
1556,Odell Beckham,WR,2021,13,LA,JAX
1557,Odell Beckham,WR,2021,14,LA,ARI


In [48]:
snap_pd[
    snap_pd['player'].str.contains('Beckham', case=False, na=False)
][[
    'player',
    'season',
    'week',
    'team',
    'offense_pct'
]].head(20)

,player,season,week,team,offense_pct
3351,Odell Beckham Jr.,2021,3,CLE,0.64
4787,Odell Beckham Jr.,2021,4,CLE,0.79
6177,Odell Beckham Jr.,2021,5,CLE,0.81
7436,Odell Beckham Jr.,2021,6,CLE,0.54
9113,Odell Beckham Jr.,2021,7,CLE,0.61
10880,Odell Beckham Jr.,2021,8,CLE,0.73
13448,Odell Beckham Jr.,2021,10,LA,0.27
15959,Odell Beckham Jr.,2021,12,LA,0.98
17213,Odell Beckham Jr.,2021,13,LA,0.53
18747,Odell Beckham Jr.,2021,14,LA,0.72


In [49]:
unmatched = model_data[
    model_data['offense_pct'].isna()
].copy()

print("Unmatched rows:", len(unmatched))
print("Unique unmatched players:", unmatched['player_display_name'].nunique())

unmatched['player_display_name'].value_counts().head(30)

Unmatched rows: 863
Unique unmatched players: 59


,count
player_display_name,
DK Metcalf,65
Michael Pittman,65
Gabe Davis,55
Brian Robinson,41
Odell Beckham,36
A.J. Dillon,34
Gardner Minshew,34
Cedrick Wilson,33
Terrace Marshall,30


In [50]:
import re
import pandas as pd

def normalize_name(name):
    if pd.isna(name):
        return name

    name = name.lower()

    # Remove punctuation: D.J. -> DJ, De'Von -> Devon
    name = re.sub(r"[.'’-]", "", name)

    # Remove common suffixes
    name = re.sub(r"\b(jr|sr|ii|iii|iv)\b", "", name)

    # Remove extra spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name


# Create normalized names in both datasets
model_data['name_norm'] = model_data['player_display_name'].apply(normalize_name)
snap_pd['name_norm'] = snap_pd['player'].apply(normalize_name)

print(model_data[['player_display_name', 'name_norm']].head())
print(snap_pd[['player', 'name_norm']].head())

  player_display_name  name_norm
0           Tom Brady  tom brady
1           Tom Brady  tom brady
2           Tom Brady  tom brady
3           Tom Brady  tom brady
4           Tom Brady  tom brady
           player       name_norm
3  Ryan Tannehill  ryan tannehill
6      A.J. Brown        aj brown
7     Julio Jones     julio jones
8   Derrick Henry   derrick henry
9  Chester Rogers  chester rogers


In [51]:
# Remove fields from our previous snap merge
columns_to_drop = [
    'player',
    'team',
    'opponent',
    'offense_snaps',
    'offense_pct'
]

model_data = model_data.drop(
    columns=[c for c in columns_to_drop if c in model_data.columns]
)

# Merge again, this time using normalized player names
model_data = model_data.merge(
    snap_pd[[
        'season',
        'week',
        'name_norm',
        'team',
        'opponent',
        'offense_snaps',
        'offense_pct'
    ]],
    left_on=[
        'season',
        'week',
        'name_norm',
        'recent_team',
        'opponent_team'
    ],
    right_on=[
        'season',
        'week',
        'name_norm',
        'team',
        'opponent'
    ],
    how='left'
)

print("Total rows:", len(model_data))
print("Matched:", model_data['offense_pct'].notna().sum())
print("Missing:", model_data['offense_pct'].isna().sum())

match_rate = model_data['offense_pct'].notna().mean() * 100
print(f"Match rate: {match_rate:.2f}%")

Total rows: 21071
Matched: 20909
Missing: 162
Match rate: 99.23%


In [52]:
unmatched = model_data[
    model_data['offense_pct'].isna()
].copy()

print("Remaining unmatched rows:", len(unmatched))
print("Remaining unique players:", unmatched['player_display_name'].nunique())

unmatched['player_display_name'].value_counts().head(30)

Remaining unmatched rows: 162
Remaining unique players: 21


,count
player_display_name,
Gabe Davis,55
Jeffery Wilson,29
Chig Okonkwo,17
Joseph Fortson,16
Mike Woods,11
Christopher Brooks,11
Christopher Herndon,5
Dee Eskridge,3
Bam Knight,2


In [53]:
names_to_check = [
    'Davis',
    'Wilson',
    'Okonkwo',
    'Fortson',
    'Woods',
    'Brooks',
    'Herndon',
    'Eskridge',
    'Knight'
]

for name in names_to_check:
    print(f"\n--- {name} ---")
    print(
        snap_pd[
            snap_pd['player'].str.contains(name, case=False, na=False)
        ]['player']
        .drop_duplicates()
        .tolist()
    )


--- Davis ---
['Corey Davis', 'Mike Davis', 'Gabriel Davis', 'Davis Mills', 'Tyler Davis', 'Davis Webb', 'Davion Davis', 'Tyrion Davis-Price', 'Malik Davis', 'Davis Allen', 'Derius Davis', 'Ray Davis', 'Isaiah Davis', 'Kaden Davis']

--- Wilson ---
['Cedrick Wilson Jr.', 'Albert Wilson', 'Zach Wilson', 'Russell Wilson', 'Jeff Wilson', 'Garrett Wilson', 'Michael Wilson', 'Emanuel Wilson', 'Johnny Wilson', 'Roman Wilson']

--- Okonkwo ---
['Chigoziem Okonkwo']

--- Fortson ---
['Jody Fortson']

--- Woods ---
['Robert Woods', 'Logan Woodside', 'Jelani Woods', 'Michael Woods II']

--- Brooks ---
['Chris Brooks', 'Jalen Brooks', 'British Brooks', 'Jonathon Brooks']

--- Herndon ---
['Chris Herndon']

--- Eskridge ---
["D'Wayne Eskridge"]

--- Knight ---
['Zonovan Knight']


In [54]:
aliases = {
    'gabe davis': 'gabriel davis',
    'jeffery wilson': 'jeff wilson',
    'chig okonkwo': 'chigoziem okonkwo',
    'joseph fortson': 'jody fortson',
    'mike woods': 'michael woods',
    'christopher brooks': 'chris brooks',
    'christopher herndon': 'chris herndon',
    'dee eskridge': 'dwayne eskridge',
    'bam knight': 'zonovan knight',
}

In [55]:
# Apply verified aliases to normalized model names
model_data['name_norm'] = model_data['name_norm'].replace(aliases)

# Remove prior snap columns before re-merging
columns_to_drop = [
    'team',
    'opponent',
    'offense_snaps',
    'offense_pct'
]

model_data = model_data.drop(
    columns=[c for c in columns_to_drop if c in model_data.columns]
)

# Merge snap data again
model_data = model_data.merge(
    snap_pd[[
        'season',
        'week',
        'name_norm',
        'team',
        'opponent',
        'offense_snaps',
        'offense_pct'
    ]],
    left_on=[
        'season',
        'week',
        'name_norm',
        'recent_team',
        'opponent_team'
    ],
    right_on=[
        'season',
        'week',
        'name_norm',
        'team',
        'opponent'
    ],
    how='left'
)

print("Total rows:", len(model_data))
print("Matched:", model_data['offense_pct'].notna().sum())
print("Missing:", model_data['offense_pct'].isna().sum())
print(f"Match rate: {model_data['offense_pct'].notna().mean() * 100:.2f}%")

Total rows: 21071
Matched: 21058
Missing: 13
Match rate: 99.94%


In [56]:
model_data[
    model_data['offense_pct'].isna()
]['player_display_name'].value_counts().head(30)

,count
player_display_name,
Giovanni Ricci,2
Elijhaa Penny,1
Brandon Powell,1
Jamal Agnew,1
Gunner Olszewski,1
Tylan Wallace,1
Ihmir Smith-Marsette,1
Rod Williams,1
Bryce Ford-Wheaton,1


In [57]:
# Make sure rows are in chronological order
model_data = model_data.sort_values(
    ['player_id', 'season', 'week']
).copy()

# Prior 3-game and 5-game snap share
model_data['offense_pct_last3'] = (
    model_data
    .groupby(['player_id', 'season'])['offense_pct']
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
)

model_data['offense_pct_last5'] = (
    model_data
    .groupby(['player_id', 'season'])['offense_pct']
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

model_data['snap_trend'] = (
    model_data['offense_pct_last3'] -
    model_data['offense_pct_last5']
)

print("Snap-share features created.")

Snap-share features created.


In [58]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

base_features = [
    'my_fantasy_points_last3',
    'carries_last3',
    'targets_last3',
    'rushing_yards_last3',
    'receiving_yards_last3',
    'target_share_last3',
    'defense_fp_allowed_last3',
    'team_spread',
    'total_line'
]

snap_features = base_features + [
    'offense_pct_last3',
    'offense_pct_last5',
    'snap_trend'
]

results = []

for pos in ['RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()
    test = pos_data[pos_data['season'] == 2024].copy()

    for label, feature_set in [
        ('Base', base_features),
        ('+ Snap share', snap_features)
    ]:

        X_train = train[feature_set].copy()
        X_test = test[feature_set].copy()

        medians = X_train.median(numeric_only=True)

        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        m = RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )

        m.fit(X_train, train['my_fantasy_points'])

        preds = m.predict(X_test)

        mae = mean_absolute_error(
            test['my_fantasy_points'],
            preds
        )

        rank = spearmanr(
            preds,
            test['my_fantasy_points']
        ).statistic

        results.append({
            'Position': pos,
            'Model': label,
            'MAE': round(mae, 3),
            'Rank correlation': round(rank, 3)
        })

pd.DataFrame(results)

,Position,Model,MAE,Rank correlation
0,RB,Base,4.659,0.591
1,RB,+ Snap share,4.671,0.601
2,WR,Base,4.484,0.531
3,WR,+ Snap share,4.430,0.541
4,TE,Base,3.389,0.494
5,TE,+ Snap share,3.344,0.504


In [59]:
qb_cols = [
    'completions',
    'attempts',
    'passing_yards',
    'passing_tds',
    'interceptions',
    'passing_air_yards',
    'passing_epa',
    'carries',
    'rushing_yards',
    'rushing_tds',
    'my_fantasy_points'
]

for col in qb_cols:
    model_data[f'{col}_last3'] = (
        model_data
        .groupby(['player_id', 'season'])[col]
        .transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    )

print("QB pregame features created.")

QB pregame features created.


In [60]:
qb_features = [
    'my_fantasy_points_last3',
    'attempts_last3',
    'completions_last3',
    'passing_yards_last3',
    'passing_tds_last3',
    'interceptions_last3',
    'passing_air_yards_last3',
    'passing_epa_last3',
    'carries_last3',
    'rushing_yards_last3',
    'rushing_tds_last3',
    'defense_fp_allowed_last3',
    'team_spread',
    'total_line',
    'is_home',
    'temp',
    'wind',
    'rest_days'
]

qb_data = model_data[
    (model_data['position'] == 'QB') &
    (model_data['my_fantasy_points_last3'].notna())
].copy()

qb_train = qb_data[qb_data['season'] <= 2023].copy()
qb_test = qb_data[qb_data['season'] == 2024].copy()

X_train = qb_train[qb_features].copy()
X_test = qb_test[qb_features].copy()

medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

qb_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

qb_model.fit(
    X_train,
    qb_train['my_fantasy_points']
)

qb_predictions = qb_model.predict(X_test)

qb_model_mae = mean_absolute_error(
    qb_test['my_fantasy_points'],
    qb_predictions
)

qb_baseline_mae = mean_absolute_error(
    qb_test['my_fantasy_points'],
    qb_test['my_fantasy_points_last3']
)

qb_model_rank = spearmanr(
    qb_predictions,
    qb_test['my_fantasy_points']
).statistic

qb_baseline_rank = spearmanr(
    qb_test['my_fantasy_points_last3'],
    qb_test['my_fantasy_points']
).statistic

print(f"2024 QB games: {len(qb_test):,}")
print()
print(f"Baseline MAE: {qb_baseline_mae:.3f}")
print(f"Model MAE:    {qb_model_mae:.3f}")
print()
print(f"Baseline rank: {qb_baseline_rank:.3f}")
print(f"Model rank:    {qb_model_rank:.3f}")

2024 QB games: 586

Baseline MAE: 6.902
Model MAE:    6.337

Baseline rank: 0.443
Model rank:    0.495


In [61]:
injuries = nfl.load_injuries([2021, 2022, 2023, 2024])

print(injuries.shape)
print(injuries.columns)
injuries.head()

(23083, 16)
['season', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status', 'date_modified']


season,game_type,team,week,gsis_id,position,full_name,first_name,last_name,report_primary_injury,report_secondary_injury,report_status,practice_primary_injury,practice_secondary_injury,practice_status,date_modified
i32,str,str,i32,str,str,str,str,str,str,str,str,str,str,str,"datetime[μs, UTC]"
2021,"""REG""","""ARI""",1,"""00-0033258""","""TE""","""Darrell Daniels""","""Darrell""","""Daniels""",null,null,null,"""Toe""",null,"""Full Participation in Practice""",2021-09-10 19:35:39 UTC
2021,"""REG""","""ARI""",1,"""00-0034473""","""LB""","""Dennis Gardeck""","""Dennis""","""Gardeck""","""Knee""","""Hand""","""Out""","""Knee""","""Hand""","""Did Not Participate In Practic…",2021-09-10 19:36:51 UTC
2021,"""REG""","""ARI""",1,"""00-0035126""","""WR""","""Antoine Wesley""","""Antoine""","""Wesley""","""Illness""",null,"""Out""","""Illness""",null,"""Did Not Participate In Practic…",2021-09-10 19:37:05 UTC
2021,"""REG""","""ATL""",1,"""00-0031583""","""DT""","""Grady Jarrett""","""Grady""","""Jarrett""",null,null,null,"""Not injury related - personal …",null,"""Did Not Participate In Practic…",2021-09-10 09:29:27 UTC
2021,"""REG""","""ATL""",1,"""00-0030010""","""LB""","""Brandon Copeland""","""Brandon""","""Copeland""",null,null,null,"""Hamstring""",null,"""Full Participation in Practice""",2021-09-10 18:05:56 UTC


In [62]:
print(injuries.shape)
print(injuries.columns)

(23083, 16)
['season', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status', 'date_modified']


In [63]:
print("Report statuses:")
print(injuries['report_status'].value_counts())

print("\nPractice statuses:")
print(injuries['practice_status'].value_counts())

Report statuses:
shape: (5, 2)
┌───────────────┬───────┐
│ report_status ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ Note          ┆ 6     │
│ null          ┆ 12221 │
│ Doubtful      ┆ 658   │
│ Questionable  ┆ 6120  │
│ Out           ┆ 4078  │
└───────────────┴───────┘

Practice statuses:
shape: (5, 2)
┌─────────────────────────────────┬───────┐
│ practice_status                 ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ Did Not Participate In Practic… ┆ 6572  │
│ Note                            ┆ 1     │
│ Full Participation in Practice  ┆ 10482 │
│                                 ┆ 141   │
│                                 ┆       │
│ Limited Participation in Pract… ┆ 5887  │
└─────────────────────────────────┴───────┘


In [64]:
print("Report statuses:")
print(injuries['report_status'].value_counts())

print("\nPractice statuses:")
print(injuries['practice_status'].value_counts())

Report statuses:
shape: (5, 2)
┌───────────────┬───────┐
│ report_status ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ Out           ┆ 4078  │
│ Questionable  ┆ 6120  │
│ Doubtful      ┆ 658   │
│ Note          ┆ 6     │
│ null          ┆ 12221 │
└───────────────┴───────┘

Practice statuses:
shape: (5, 2)
┌─────────────────────────────────┬───────┐
│ practice_status                 ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ Limited Participation in Pract… ┆ 5887  │
│ Full Participation in Practice  ┆ 10482 │
│                                 ┆ 141   │
│                                 ┆       │
│ Did Not Participate In Practic… ┆ 6572  │
│ Note                            ┆ 1     │
└─────────────────────────────────┴───────┘


In [65]:
injuries.filter(
    injuries['full_name'].str.contains('Jonathan Taylor')
).select([
    'season',
    'week',
    'team',
    'full_name',
    'report_primary_injury',
    'report_status',
    'practice_primary_injury',
    'practice_status',
    'date_modified'
]).head(30)

season,week,team,full_name,report_primary_injury,report_status,practice_primary_injury,practice_status,date_modified
i32,i32,str,str,str,str,str,str,"datetime[μs, UTC]"
2021,4,"""IND""","""Jonathan Taylor""","""Knee""","""Questionable""","""Knee""","""Limited Participation in Pract…",2021-10-01 18:30:08 UTC
2021,8,"""IND""","""Jonathan Taylor""",null,null,"""Ribs""","""Full Participation in Practice""",2021-10-29 18:35:33 UTC
2022,4,"""IND""","""Jonathan Taylor""",null,null,"""Toe""","""Full Participation in Practice""",2022-09-30 19:38:02 UTC
2022,5,"""IND""","""Jonathan Taylor""","""Ankle""","""Out""","""Ankle""","""Did Not Participate In Practic…",2022-10-05 17:14:57 UTC
2022,6,"""IND""","""Jonathan Taylor""","""Ankle""","""Questionable""","""Ankle""","""Limited Participation in Pract…",2022-10-14 19:28:44 UTC
…,…,…,…,…,…,…,…,…
2023,16,"""IND""","""Jonathan Taylor""",null,null,"""Thumb""","""Full Participation in Practice""",2023-12-22 20:28:09 UTC
2024,5,"""IND""","""Jonathan Taylor""","""Ankle""","""Out""","""Ankle""","""Did Not Participate In Practic…",2024-10-04 14:05:26 UTC
2024,6,"""IND""","""Jonathan Taylor""","""Ankle""","""Out""","""Ankle""","""Did Not Participate In Practic…",2024-10-11 12:45:47 UTC


In [66]:
injury_pd = injuries.to_pandas()

In [67]:
id_check = model_data[
    ['player_display_name', 'player_id']
].drop_duplicates()

injury_id_check = injury_pd[
    ['full_name', 'gsis_id']
].drop_duplicates()

id_matches = id_check.merge(
    injury_id_check,
    left_on='player_id',
    right_on='gsis_id',
    how='inner'
)

print("Unique model players:", id_check['player_id'].nunique())
print("Players matched by ID:", id_matches['player_id'].nunique())

id_matches[
    ['player_display_name', 'player_id', 'full_name', 'gsis_id']
].head(20)

Unique model players: 961
Players matched by ID: 760


,player_display_name,player_id,full_name,gsis_id
0,Tom Brady,00-0019596,Tom Brady,00-0019596
1,Ben Roethlisberger,00-0022924,Ben Roethlisberger,00-0022924
2,Aaron Rodgers,00-0023459,Aaron Rodgers,00-0023459
3,Ryan Fitzpatrick,00-0023682,Ryan Fitzpatrick,00-0023682
4,Marcedes Lewis,00-0024243,Marcedes Lewis,00-0024243
5,Danny Amendola,00-0026035,Danny Amendola,00-0026035
6,Matt Ryan,00-0026143,Matt Ryan,00-0026143
7,Joe Flacco,00-0026158,Joe Flacco,00-0026158
8,DeSean Jackson,00-0026189,DeSean Jackson,00-0026189
9,Chad Henne,00-0026197,Chad Henne,00-0026197


In [68]:
# Keep the fields we need
injury_features = injury_pd[[
    'season',
    'week',
    'team',
    'gsis_id',
    'report_status',
    'practice_status'
]].copy()

# Convert statuses into model-friendly flags
injury_features['on_injury_report'] = 1

injury_features['questionable'] = (
    injury_features['report_status'] == 'Questionable'
).astype(int)

injury_features['doubtful'] = (
    injury_features['report_status'] == 'Doubtful'
).astype(int)

injury_features['out'] = (
    injury_features['report_status'] == 'Out'
).astype(int)

injury_features['practice_full'] = (
    injury_features['practice_status'] == 'Full Participation in Practice'
).astype(int)

injury_features['practice_limited'] = (
    injury_features['practice_status'] == 'Limited Participation in Practice'
).astype(int)

injury_features['practice_dnp'] = (
    injury_features['practice_status'] == 'Did Not Participate In Practice'
).astype(int)

# Keep one record per player/week/team
injury_features = injury_features.drop_duplicates(
    subset=['season', 'week', 'team', 'gsis_id']
)

injury_features.head()

,season,week,team,gsis_id,report_status,practice_status,on_injury_report,questionable,doubtful,out,practice_full,practice_limited,practice_dnp
0,2021,1,ARI,00-0033258,None,Full Participation in Practice,1,0,0,0,1,0,0
1,2021,1,ARI,00-0034473,Out,Did Not Participate In Practice,1,0,0,1,0,0,1
2,2021,1,ARI,00-0035126,Out,Did Not Participate In Practice,1,0,0,1,0,0,1
3,2021,1,ATL,00-0031583,None,Did Not Participate In Practice,1,0,0,0,0,0,1
4,2021,1,ATL,00-0030010,None,Full Participation in Practice,1,0,0,0,1,0,0


In [69]:
# Remove leftover helper columns from earlier merges
helper_cols = [
    'team',
    'team_x',
    'team_y',
    'opponent',
    'opponent_x',
    'opponent_y',
    'gsis_id',
    'injury_team',
    'injury_player_id'
]

model_data = model_data.drop(
    columns=[c for c in helper_cols if c in model_data.columns]
)

# Prepare injury data with unique helper names
injury_merge = injury_features[[
    'season',
    'week',
    'team',
    'gsis_id',
    'on_injury_report',
    'questionable',
    'doubtful',
    'out',
    'practice_full',
    'practice_limited',
    'practice_dnp'
]].copy()

injury_merge = injury_merge.rename(columns={
    'team': 'injury_team',
    'gsis_id': 'injury_player_id'
})

# Merge injury features onto model_data
model_data = model_data.merge(
    injury_merge,
    left_on=[
        'season',
        'week',
        'recent_team',
        'player_id'
    ],
    right_on=[
        'season',
        'week',
        'injury_team',
        'injury_player_id'
    ],
    how='left'
)

injury_cols = [
    'on_injury_report',
    'questionable',
    'doubtful',
    'out',
    'practice_full',
    'practice_limited',
    'practice_dnp'
]

model_data[injury_cols] = model_data[injury_cols].fillna(0)

print("Injury features attached.")
print(model_data[injury_cols].sum())

Injury features attached.
on_injury_report    3781.0
questionable         988.0
doubtful               1.0
out                    0.0
practice_full       2325.0
practice_limited    1101.0
practice_dnp         336.0
dtype: float64


In [70]:
# Remove leftover helper columns from earlier merges
helper_cols = [
    'team',
    'team_x',
    'team_y',
    'opponent',
    'opponent_x',
    'opponent_y',
    'gsis_id'
]

model_data = model_data.drop(
    columns=[c for c in helper_cols if c in model_data.columns]
)

print("Helper columns removed.")

Helper columns removed.


In [71]:
# Injury feature column names
injury_cols = [
    'on_injury_report',
    'questionable',
    'doubtful',
    'out',
    'practice_full',
    'practice_limited',
    'practice_dnp'
]

# Remove any leftovers from prior runs/merges
columns_to_remove = [
    'team',
    'team_x',
    'team_y',
    'opponent',
    'opponent_x',
    'opponent_y',
    'gsis_id',
    'injury_team',
    'injury_player_id',

    # Old injury columns
    'on_injury_report',
    'questionable',
    'doubtful',
    'out',
    'practice_full',
    'practice_limited',
    'practice_dnp',

    # Possible suffixed versions from failed/repeated merges
    'on_injury_report_x',
    'on_injury_report_y',
    'questionable_x',
    'questionable_y',
    'doubtful_x',
    'doubtful_y',
    'out_x',
    'out_y',
    'practice_full_x',
    'practice_full_y',
    'practice_limited_x',
    'practice_limited_y',
    'practice_dnp_x',
    'practice_dnp_y'
]

model_data = model_data.drop(
    columns=[c for c in columns_to_remove if c in model_data.columns]
)

# Prepare injury table
injury_merge = injury_features[[
    'season',
    'week',
    'team',
    'gsis_id',
    'on_injury_report',
    'questionable',
    'doubtful',
    'out',
    'practice_full',
    'practice_limited',
    'practice_dnp'
]].copy()

injury_merge = injury_merge.rename(columns={
    'team': 'injury_team',
    'gsis_id': 'injury_player_id'
})

# Merge cleanly
model_data = model_data.merge(
    injury_merge,
    left_on=[
        'season',
        'week',
        'recent_team',
        'player_id'
    ],
    right_on=[
        'season',
        'week',
        'injury_team',
        'injury_player_id'
    ],
    how='left'
)

# Players not listed that week get zeros
model_data[injury_cols] = model_data[injury_cols].fillna(0)

print("Injury features attached.")
print(model_data[injury_cols].sum())

Injury features attached.
on_injury_report    3781.0
questionable         988.0
doubtful               1.0
out                    0.0
practice_full       2325.0
practice_limited    1101.0
practice_dnp         336.0
dtype: float64


In [72]:
injury_cols = [
    'on_injury_report',
    'questionable',
    'practice_full',
    'practice_limited',
    'practice_dnp'
]

position_features = {

    'QB': [
        'my_fantasy_points_last3',
        'attempts_last3',
        'completions_last3',
        'passing_yards_last3',
        'passing_tds_last3',
        'interceptions_last3',
        'passing_air_yards_last3',
        'passing_epa_last3',
        'carries_last3',
        'rushing_yards_last3',
        'rushing_tds_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'is_home',
        'temp',
        'wind',
        'rest_days'
    ],

    'RB': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend'
    ],

    'WR': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend'
    ],

    'TE': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend'
    ]
}

In [73]:
results = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()
    test = pos_data[pos_data['season'] == 2024].copy()

    for label, cols in [
        ('Without injuries', position_features[pos]),
        ('+ Injuries', position_features[pos] + injury_cols)
    ]:

        X_train = train[cols].copy()
        X_test = test[cols].copy()

        medians = X_train.median(numeric_only=True)

        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        m = RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )

        m.fit(
            X_train,
            train['my_fantasy_points']
        )

        preds = m.predict(X_test)

        mae = mean_absolute_error(
            test['my_fantasy_points'],
            preds
        )

        rank = spearmanr(
            preds,
            test['my_fantasy_points']
        ).statistic

        results.append({
            'Position': pos,
            'Model': label,
            'MAE': round(mae, 3),
            'Rank correlation': round(rank, 3)
        })

pd.DataFrame(results)

,Position,Model,MAE,Rank correlation
0,QB,Without injuries,6.337,0.495
1,QB,+ Injuries,6.341,0.494
2,RB,Without injuries,4.671,0.601
3,RB,+ Injuries,4.671,0.602
4,WR,Without injuries,4.430,0.541
5,WR,+ Injuries,4.425,0.542
6,TE,Without injuries,3.344,0.504
7,TE,+ Injuries,3.342,0.505


In [74]:
participation = nfl.load_participation([2021, 2022, 2023, 2024])

print(participation.shape)
print(participation.columns)

(192951, 26)
['nflverse_game_id', 'old_game_id', 'play_id', 'possession_team', 'offense_formation', 'offense_personnel', 'defenders_in_box', 'defense_personnel', 'number_of_pass_rushers', 'players_on_play', 'offense_players', 'defense_players', 'n_offense', 'n_defense', 'ngs_air_yards', 'time_to_throw', 'was_pressure', 'route', 'defense_man_zone_type', 'defense_coverage_type', 'offense_names', 'defense_names', 'offense_positions', 'defense_positions', 'offense_numbers', 'defense_numbers']


In [75]:
participation_pd = participation.to_pandas()

# Derive season and week from nflverse_game_id
participation_pd['season'] = (
    participation_pd['nflverse_game_id']
    .str.slice(0, 4)
    .astype(int)
)

participation_pd['week'] = (
    participation_pd['nflverse_game_id']
    .str.extract(r'_(\d+)_')[0]
    .astype(int)
)

participation_pd[[
    'season',
    'week',
    'nflverse_game_id',
    'play_id',
    'possession_team',
    'offense_formation',
    'offense_personnel',
    'offense_players',
    'offense_names',
    'offense_positions',
    'route'
]].head(20)

,season,week,nflverse_game_id,play_id,possession_team,offense_formation,offense_personnel,offense_players,offense_names,offense_positions,route
0,2021,1,2021_01_DAL_TB,1.0,,None,None,,None,None,None
1,2021,1,2021_01_DAL_TB,40.0,DAL,None,None,00-0033083;00-0033512;00-0034674;00-0036882;00...,None,None,None
2,2021,1,2021_01_DAL_TB,55.0,TB,SHOTGUN,"1 RB, 1 TE, 3 WR",00-0033921;00-0032243;00-0036406;00-0019596;00...,None,None,None
3,2021,1,2021_01_DAL_TB,76.0,TB,SINGLEBACK,"1 RB, 2 TE, 2 WR",00-0033921;00-0032243;00-0036406;00-0019596;00...,None,None,None
4,2021,1,2021_01_DAL_TB,97.0,TB,SHOTGUN,"1 RB, 1 TE, 3 WR",00-0033921;00-0032243;00-0036406;00-0019596;00...,None,None,CROSS
5,2021,1,2021_01_DAL_TB,119.0,TB,None,None,00-0036643;00-0033595;00-0034515;00-0032065;00...,None,None,None
6,2021,1,2021_01_DAL_TB,137.0,DAL,EMPTY,"1 RB, 2 TE, 2 WR",00-0033077;00-0035679;00-0034352;00-0036358;00...,None,None,GO
7,2021,1,2021_01_DAL_TB,166.0,DAL,SINGLEBACK,"1 RB, 1 TE, 3 WR",00-0033077;00-0035679;00-0034764;00-0034352;00...,None,None,None
8,2021,1,2021_01_DAL_TB,187.0,DAL,SHOTGUN,"0 RB, 2 TE, 3 WR",00-0033077;00-0035679;00-0034764;00-0034352;00...,None,None,HITCH
9,2021,1,2021_01_DAL_TB,211.0,DAL,I_FORM,"2 RB, 2 TE, 1 WR",00-0033077;00-0035679;00-0034418;00-0034352;00...,None,None,None


In [76]:
participation_pd = participation.to_pandas()

participation_pd['season'] = (
    participation_pd['nflverse_game_id']
    .str.slice(0, 4)
    .astype(int)
)

participation_pd['week'] = (
    participation_pd['nflverse_game_id']
    .str.extract(r'_(\d+)_')[0]
    .astype(int)
)

participation_pd[[
    'season',
    'week',
    'nflverse_game_id',
    'play_id',
    'possession_team',
    'offense_formation',
    'offense_personnel',
    'offense_players',
    'offense_names',
    'offense_positions',
    'route'
]].head(20)

,season,week,nflverse_game_id,play_id,possession_team,offense_formation,offense_personnel,offense_players,offense_names,offense_positions,route
0,2021,1,2021_01_DAL_TB,1.0,,None,None,,None,None,None
1,2021,1,2021_01_DAL_TB,40.0,DAL,None,None,00-0033083;00-0033512;00-0034674;00-0036882;00...,None,None,None
2,2021,1,2021_01_DAL_TB,55.0,TB,SHOTGUN,"1 RB, 1 TE, 3 WR",00-0033921;00-0032243;00-0036406;00-0019596;00...,None,None,None
3,2021,1,2021_01_DAL_TB,76.0,TB,SINGLEBACK,"1 RB, 2 TE, 2 WR",00-0033921;00-0032243;00-0036406;00-0019596;00...,None,None,None
4,2021,1,2021_01_DAL_TB,97.0,TB,SHOTGUN,"1 RB, 1 TE, 3 WR",00-0033921;00-0032243;00-0036406;00-0019596;00...,None,None,CROSS
5,2021,1,2021_01_DAL_TB,119.0,TB,None,None,00-0036643;00-0033595;00-0034515;00-0032065;00...,None,None,None
6,2021,1,2021_01_DAL_TB,137.0,DAL,EMPTY,"1 RB, 2 TE, 2 WR",00-0033077;00-0035679;00-0034352;00-0036358;00...,None,None,GO
7,2021,1,2021_01_DAL_TB,166.0,DAL,SINGLEBACK,"1 RB, 1 TE, 3 WR",00-0033077;00-0035679;00-0034764;00-0034352;00...,None,None,None
8,2021,1,2021_01_DAL_TB,187.0,DAL,SHOTGUN,"0 RB, 2 TE, 3 WR",00-0033077;00-0035679;00-0034764;00-0034352;00...,None,None,HITCH
9,2021,1,2021_01_DAL_TB,211.0,DAL,I_FORM,"2 RB, 2 TE, 1 WR",00-0033077;00-0035679;00-0034418;00-0034352;00...,None,None,None


In [77]:
print(type(participation_pd.loc[0, 'offense_players']))
print(participation_pd.loc[0, 'offense_players'])
print()
print(participation_pd.loc[0, 'offense_names'])
print()
print(participation_pd.loc[0, 'offense_positions'])

<class 'str'>


None

None


In [78]:
sample = participation_pd[
    participation_pd['offense_players'].notna()
].head(1)

sample[[
    'season',
    'week',
    'nflverse_game_id',
    'play_id',
    'possession_team',
    'offense_formation',
    'offense_personnel',
    'offense_players',
    'offense_names',
    'offense_positions',
    'route'
]]

,season,week,nflverse_game_id,play_id,possession_team,offense_formation,offense_personnel,offense_players,offense_names,offense_positions,route
0,2021,1,2021_01_DAL_TB,1.0,,None,None,,None,None,None


In [79]:
row = sample.iloc[0]

print(type(row['offense_players']))
print(row['offense_players'])
print()
print(row['offense_names'])
print()
print(row['offense_positions'])

<class 'str'>


None

None


In [80]:
print("Non-null offense_players:",
      participation_pd['offense_players'].notna().sum())

print("Non-null offense_names:",
      participation_pd['offense_names'].notna().sum())

print("Non-null offense_positions:",
      participation_pd['offense_positions'].notna().sum())

print("Non-null route:",
      participation_pd['route'].notna().sum())

Non-null offense_players: 192951
Non-null offense_names: 92087
Non-null offense_positions: 92087
Non-null route: 129186


In [81]:
participation_pd[
    participation_pd['offense_names'].notna()
][[
    'season',
    'week',
    'possession_team',
    'offense_players',
    'offense_names',
    'offense_positions',
    'route'
]].head(10)

,season,week,possession_team,offense_players,offense_names,offense_positions,route
100864,2023,1,WAS,00-0037095;00-0039149;00-0036403;00-0038635;00...,Christian Holmes;Jartavius Martin;Casey Toohil...,CB;CB;DE;DE;FS;MLB;MLB;OLB;RB;RB;TE,
100865,2023,1,WAS,00-0034445;00-0033831;00-0037077;00-0037746;00...,Nick Gates;Andrew Wylie;Sam Howell;Brian Robin...,C;G;QB;RB;T;T;T;TE;TE;WR;WR,
100866,2023,1,WAS,00-0034445;00-0033831;00-0037077;00-0037746;00...,Nick Gates;Andrew Wylie;Sam Howell;Brian Robin...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,HITCH/CURL
100867,2023,1,WAS,00-0034445;00-0033831;00-0037077;00-0038611;00...,Nick Gates;Andrew Wylie;Sam Howell;Chris Rodri...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,
100868,2023,1,WAS,00-0034445;00-0033831;00-0037077;00-0037746;00...,Nick Gates;Andrew Wylie;Sam Howell;Brian Robin...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,IN/DIG
100869,2023,1,WAS,00-0034445;00-0033831;00-0037077;00-0037746;00...,Nick Gates;Andrew Wylie;Sam Howell;Brian Robin...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,HITCH/CURL
100870,2023,1,WAS,00-0034445;00-0033831;00-0037077;00-0037746;00...,Nick Gates;Andrew Wylie;Sam Howell;Brian Robin...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,SCREEN
100871,2023,1,WAS,00-0034445;00-0033831;00-0037077;00-0036328;00...,Nick Gates;Andrew Wylie;Sam Howell;Antonio Gib...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,QUICK OUT
100872,2023,1,WAS,00-0034445;00-0033831;00-0037077;00-0036328;00...,Nick Gates;Andrew Wylie;Sam Howell;Antonio Gib...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,QUICK OUT
100873,2023,1,WAS,00-0037095;00-0039149;00-0036403;00-0038635;00...,Christian Holmes;Jartavius Martin;Casey Toohil...,CB;CB;DE;DE;FS;LS;MLB;MLB;P;RB;TE,


In [82]:
ff_opportunity = nfl.load_ff_opportunity([2021, 2022, 2023, 2024])

print(ff_opportunity.shape)
print(ff_opportunity.columns)
ff_opportunity.head()

(24189, 159)
['season', 'posteam', 'week', 'game_id', 'player_id', 'full_name', 'position', 'pass_attempt', 'rec_attempt', 'rush_attempt', 'pass_air_yards', 'rec_air_yards', 'pass_completions', 'receptions', 'pass_completions_exp', 'receptions_exp', 'pass_yards_gained', 'rec_yards_gained', 'rush_yards_gained', 'pass_yards_gained_exp', 'rec_yards_gained_exp', 'rush_yards_gained_exp', 'pass_touchdown', 'rec_touchdown', 'rush_touchdown', 'pass_touchdown_exp', 'rec_touchdown_exp', 'rush_touchdown_exp', 'pass_two_point_conv', 'rec_two_point_conv', 'rush_two_point_conv', 'pass_two_point_conv_exp', 'rec_two_point_conv_exp', 'rush_two_point_conv_exp', 'pass_first_down', 'rec_first_down', 'rush_first_down', 'pass_first_down_exp', 'rec_first_down_exp', 'rush_first_down_exp', 'pass_interception', 'rec_interception', 'pass_interception_exp', 'rec_interception_exp', 'rec_fumble_lost', 'rush_fumble_lost', 'pass_fantasy_points_exp', 'rec_fantasy_points_exp', 'rush_fantasy_points_exp', 'pass_fantasy_p

season,posteam,week,game_id,player_id,full_name,position,pass_attempt,rec_attempt,rush_attempt,pass_air_yards,rec_air_yards,pass_completions,receptions,pass_completions_exp,receptions_exp,pass_yards_gained,rec_yards_gained,rush_yards_gained,pass_yards_gained_exp,rec_yards_gained_exp,rush_yards_gained_exp,pass_touchdown,rec_touchdown,rush_touchdown,pass_touchdown_exp,rec_touchdown_exp,rush_touchdown_exp,pass_two_point_conv,rec_two_point_conv,rush_two_point_conv,pass_two_point_conv_exp,rec_two_point_conv_exp,rush_two_point_conv_exp,pass_first_down,rec_first_down,rush_first_down,…,pass_fantasy_points_exp_team,rec_fantasy_points_exp_team,rush_fantasy_points_exp_team,pass_fantasy_points_team,rec_fantasy_points_team,rush_fantasy_points_team,pass_completions_diff_team,receptions_diff_team,pass_yards_gained_diff_team,rec_yards_gained_diff_team,rush_yards_gained_diff_team,pass_touchdown_diff_team,rec_touchdown_diff_team,rush_touchdown_diff_team,pass_two_point_conv_diff_team,rec_two_point_conv_diff_team,rush_two_point_conv_diff_team,pass_first_down_diff_team,rec_first_down_diff_team,rush_first_down_diff_team,pass_interception_diff_team,rec_interception_diff_team,pass_fantasy_points_diff_team,rec_fantasy_points_diff_team,rush_fantasy_points_diff_team,total_yards_gained_team,total_yards_gained_exp_team,total_yards_gained_diff_team,total_touchdown_team,total_touchdown_exp_team,total_touchdown_diff_team,total_first_down_team,total_first_down_exp_team,total_first_down_diff_team,total_fantasy_points_team,total_fantasy_points_exp_team,total_fantasy_points_diff_team
str,str,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2021""","""TEN""",1.0,"""2021_01_ARI_TEN""","""00-0029701""","""Ryan Tannehill""","""QB""",35.0,0.0,2.0,270.0,0.0,21.0,0.0,23.61,0.0,212.0,0.0,17.0,257.23,0.0,7.32,1.0,0.0,1.0,1.74,0.0,0.68,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,2.0,…,15.72,61.32,20.87,10.72,49.8,14.6,-2.38,-2.38,-46.98,-46.98,-6.79,-0.74,-0.73,-0.94,0.0,0.0,0.0,-2.1,-2.1,-1.8,0.09,0.08,-5.0,-11.52,-6.27,304.0,357.77,-53.77,2.0,3.67,-1.67,17.0,20.9,-3.9,64.4,82.19,-17.79
"""2021""","""TEN""",1.0,"""2021_01_ARI_TEN""","""00-0032764""","""Derrick Henry""","""RB""",0.0,4.0,17.0,0.0,6.0,0.0,3.0,0.0,3.28,0.0,19.0,58.0,0.0,26.69,70.64,0.0,0.0,0.0,0.0,0.01,1.24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,…,15.72,61.32,20.87,10.72,49.8,14.6,-2.38,-2.38,-46.98,-46.98,-6.79,-0.74,-0.73,-0.94,0.0,0.0,0.0,-2.1,-2.1,-1.8,0.09,0.08,-5.0,-11.52,-6.27,304.0,357.77,-53.77,2.0,3.67,-1.67,17.0,20.9,-3.9,64.4,82.19,-17.79
"""2021""","""TEN""",1.0,"""2021_01_ARI_TEN""","""00-0032355""","""Chester Rogers""","""WR""",0.0,6.0,0.0,0.0,61.0,0.0,4.0,0.0,4.0,0.0,62.0,0.0,0.0,52.54,0.0,0.0,0.0,0.0,0.0,0.04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,…,15.72,61.32,20.87,10.72,49.8,14.6,-2.38,-2.38,-46.98,-46.98,-6.79,-0.74,-0.73,-0.94,0.0,0.0,0.0,-2.1,-2.1,-1.8,0.09,0.08,-5.0,-11.52,-6.27,304.0,357.77,-53.77,2.0,3.67,-1.67,17.0,20.9,-3.9,64.4,82.19,-17.79
"""2021""","""ARI""",1.0,"""2021_01_ARI_TEN""","""00-0035228""","""Kyler Murray""","""QB""",32.0,0.0,5.0,262.0,0.0,21.0,0.0,21.03,0.0,289.0,0.0,20.0,248.8,0.0,25.29,4.0,0.0,1.0,2.74,0.0,0.66,0.0,0.0,0.0,0.0,0.0,0.0,13.0,0.0,2.0,…,19.73,62.34,21.91,25.56,73.9,19.6,-0.03,-0.04,40.2,40.22,-12.13,1.26,1.26,-0.19,0.0,0.0,0.0,0.08,0.07,0.18,0.41,0.41,5.83,11.56,-2.31,425.0,396.91,28.09,5.0,3.93,1.07,20.0,19.75,0.25,93.5,84.25,9.25
"""2021""","""ARI""",1.0,"""2021_01_ARI_TEN""","""00-0030564""","""DeAndre Hopkins""","""WR""",0.0,8.0,0.0,0.0,90.0,0.0,6.0,0.0,4.99,0.0,83.0,0.0,0.0,69.68,0.0,0.0,2.0,0.0,0.0,0.45,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,…,19.73,62.34,21.91,25.56,73.9,19.6,-0.03,-0.04,40.2,40.22,-12.13,1.26,1.26,-0.19,0.0,0.0,0.0,0.08,0.07,0.18,0.41,0.41,5.83,11.56,-2.31,425.0,396.91,28.09,5.0,3.93,1.07,20.0,19.7

In [83]:
import nflreadpy as nfl

In [84]:
!pip install nflreadpy -q

In [85]:
import nflreadpy as nfl

print("nflreadpy loaded successfully")

nflreadpy loaded successfully


In [86]:
ff_opportunity = nfl.load_ff_opportunity([2021, 2022, 2023, 2024])

print(ff_opportunity.shape)
print(ff_opportunity.columns)
ff_opportunity.head()

(24189, 159)
['season', 'posteam', 'week', 'game_id', 'player_id', 'full_name', 'position', 'pass_attempt', 'rec_attempt', 'rush_attempt', 'pass_air_yards', 'rec_air_yards', 'pass_completions', 'receptions', 'pass_completions_exp', 'receptions_exp', 'pass_yards_gained', 'rec_yards_gained', 'rush_yards_gained', 'pass_yards_gained_exp', 'rec_yards_gained_exp', 'rush_yards_gained_exp', 'pass_touchdown', 'rec_touchdown', 'rush_touchdown', 'pass_touchdown_exp', 'rec_touchdown_exp', 'rush_touchdown_exp', 'pass_two_point_conv', 'rec_two_point_conv', 'rush_two_point_conv', 'pass_two_point_conv_exp', 'rec_two_point_conv_exp', 'rush_two_point_conv_exp', 'pass_first_down', 'rec_first_down', 'rush_first_down', 'pass_first_down_exp', 'rec_first_down_exp', 'rush_first_down_exp', 'pass_interception', 'rec_interception', 'pass_interception_exp', 'rec_interception_exp', 'rec_fumble_lost', 'rush_fumble_lost', 'pass_fantasy_points_exp', 'rec_fantasy_points_exp', 'rush_fantasy_points_exp', 'pass_fantasy_p

season,posteam,week,game_id,player_id,full_name,position,pass_attempt,rec_attempt,rush_attempt,pass_air_yards,rec_air_yards,pass_completions,receptions,pass_completions_exp,receptions_exp,pass_yards_gained,rec_yards_gained,rush_yards_gained,pass_yards_gained_exp,rec_yards_gained_exp,rush_yards_gained_exp,pass_touchdown,rec_touchdown,rush_touchdown,pass_touchdown_exp,rec_touchdown_exp,rush_touchdown_exp,pass_two_point_conv,rec_two_point_conv,rush_two_point_conv,pass_two_point_conv_exp,rec_two_point_conv_exp,rush_two_point_conv_exp,pass_first_down,rec_first_down,rush_first_down,…,pass_fantasy_points_exp_team,rec_fantasy_points_exp_team,rush_fantasy_points_exp_team,pass_fantasy_points_team,rec_fantasy_points_team,rush_fantasy_points_team,pass_completions_diff_team,receptions_diff_team,pass_yards_gained_diff_team,rec_yards_gained_diff_team,rush_yards_gained_diff_team,pass_touchdown_diff_team,rec_touchdown_diff_team,rush_touchdown_diff_team,pass_two_point_conv_diff_team,rec_two_point_conv_diff_team,rush_two_point_conv_diff_team,pass_first_down_diff_team,rec_first_down_diff_team,rush_first_down_diff_team,pass_interception_diff_team,rec_interception_diff_team,pass_fantasy_points_diff_team,rec_fantasy_points_diff_team,rush_fantasy_points_diff_team,total_yards_gained_team,total_yards_gained_exp_team,total_yards_gained_diff_team,total_touchdown_team,total_touchdown_exp_team,total_touchdown_diff_team,total_first_down_team,total_first_down_exp_team,total_first_down_diff_team,total_fantasy_points_team,total_fantasy_points_exp_team,total_fantasy_points_diff_team
str,str,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2021""","""TEN""",1.0,"""2021_01_ARI_TEN""","""00-0029701""","""Ryan Tannehill""","""QB""",35.0,0.0,2.0,270.0,0.0,21.0,0.0,23.61,0.0,212.0,0.0,17.0,257.23,0.0,7.32,1.0,0.0,1.0,1.74,0.0,0.68,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,2.0,…,15.72,61.32,20.87,10.72,49.8,14.6,-2.38,-2.38,-46.98,-46.98,-6.79,-0.74,-0.73,-0.94,0.0,0.0,0.0,-2.1,-2.1,-1.8,0.09,0.08,-5.0,-11.52,-6.27,304.0,357.77,-53.77,2.0,3.67,-1.67,17.0,20.9,-3.9,64.4,82.19,-17.79
"""2021""","""TEN""",1.0,"""2021_01_ARI_TEN""","""00-0032764""","""Derrick Henry""","""RB""",0.0,4.0,17.0,0.0,6.0,0.0,3.0,0.0,3.28,0.0,19.0,58.0,0.0,26.69,70.64,0.0,0.0,0.0,0.0,0.01,1.24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,…,15.72,61.32,20.87,10.72,49.8,14.6,-2.38,-2.38,-46.98,-46.98,-6.79,-0.74,-0.73,-0.94,0.0,0.0,0.0,-2.1,-2.1,-1.8,0.09,0.08,-5.0,-11.52,-6.27,304.0,357.77,-53.77,2.0,3.67,-1.67,17.0,20.9,-3.9,64.4,82.19,-17.79
"""2021""","""TEN""",1.0,"""2021_01_ARI_TEN""","""00-0032355""","""Chester Rogers""","""WR""",0.0,6.0,0.0,0.0,61.0,0.0,4.0,0.0,4.0,0.0,62.0,0.0,0.0,52.54,0.0,0.0,0.0,0.0,0.0,0.04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,…,15.72,61.32,20.87,10.72,49.8,14.6,-2.38,-2.38,-46.98,-46.98,-6.79,-0.74,-0.73,-0.94,0.0,0.0,0.0,-2.1,-2.1,-1.8,0.09,0.08,-5.0,-11.52,-6.27,304.0,357.77,-53.77,2.0,3.67,-1.67,17.0,20.9,-3.9,64.4,82.19,-17.79
"""2021""","""ARI""",1.0,"""2021_01_ARI_TEN""","""00-0035228""","""Kyler Murray""","""QB""",32.0,0.0,5.0,262.0,0.0,21.0,0.0,21.03,0.0,289.0,0.0,20.0,248.8,0.0,25.29,4.0,0.0,1.0,2.74,0.0,0.66,0.0,0.0,0.0,0.0,0.0,0.0,13.0,0.0,2.0,…,19.73,62.34,21.91,25.56,73.9,19.6,-0.03,-0.04,40.2,40.22,-12.13,1.26,1.26,-0.19,0.0,0.0,0.0,0.08,0.07,0.18,0.41,0.41,5.83,11.56,-2.31,425.0,396.91,28.09,5.0,3.93,1.07,20.0,19.75,0.25,93.5,84.25,9.25
"""2021""","""ARI""",1.0,"""2021_01_ARI_TEN""","""00-0030564""","""DeAndre Hopkins""","""WR""",0.0,8.0,0.0,0.0,90.0,0.0,6.0,0.0,4.99,0.0,83.0,0.0,0.0,69.68,0.0,0.0,2.0,0.0,0.0,0.45,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,…,19.73,62.34,21.91,25.56,73.9,19.6,-0.03,-0.04,40.2,40.22,-12.13,1.26,1.26,-0.19,0.0,0.0,0.0,0.08,0.07,0.18,0.41,0.41,5.83,11.56,-2.31,425.0,396.91,28.09,5.0,3.93,1.07,20.0,19.7

In [87]:
ff_opp_pd = ff_opportunity.to_pandas()

# Week came through as a float, so clean that up
ff_opp_pd['week'] = ff_opp_pd['week'].astype(int)

# Expected fantasy value of the opportunities,
# translated into YOUR league scoring.
#
# We intentionally do NOT include yardage bonuses here.
# This is being used as an input/feature, not as the final fantasy score.

ff_opp_pd['my_expected_opportunity_points'] = (

    # Passing
    (ff_opp_pd['pass_completions_exp'] * -0.1) +
    (ff_opp_pd['pass_yards_gained_exp'] / 30) +
    (ff_opp_pd['pass_touchdown_exp'] * 5) +
    (ff_opp_pd['pass_interception_exp'] * -2) +

    # Rushing
    (ff_opp_pd['rush_yards_gained_exp'] / 10) +
    (ff_opp_pd['rush_touchdown_exp'] * 6) +

    # Receiving
    (ff_opp_pd['receptions_exp'] * 0.5) +
    (ff_opp_pd['rec_yards_gained_exp'] / 10) +
    (ff_opp_pd['rec_touchdown_exp'] * 6) +

    # 2-point conversions
    (
        ff_opp_pd['pass_two_point_conv_exp'] +
        ff_opp_pd['rush_two_point_conv_exp'] +
        ff_opp_pd['rec_two_point_conv_exp']
    ) * 2
)

ff_opp_pd[[
    'full_name',
    'position',
    'season',
    'week',
    'posteam',
    'my_expected_opportunity_points',
    'total_fantasy_points_exp'
]].head(20)

,full_name,position,season,week,posteam,my_expected_opportunity_points,total_fantasy_points_exp
0,Ryan Tannehill,QB,2021,1,TEN,17.925333,20.25
1,Derrick Henry,RB,2021,1,TEN,18.873000,20.51
2,Chester Rogers,WR,2021,1,TEN,7.494000,9.49
3,Kyler Murray,QB,2021,1,ARI,25.199333,26.20
4,DeAndre Hopkins,WR,2021,1,ARI,12.163000,14.65
5,Chase Edmonds,RB,2021,1,ARI,11.187000,12.88
6,A.J. Green,WR,2021,1,ARI,9.334000,11.18
7,Christian Kirk,WR,2021,1,ARI,9.721000,11.19
8,Rondale Moore,WR,2021,1,ARI,7.424000,9.23
9,Demetrius Harris,TE,2021,1,ARI,6.536000,7.49


In [88]:
# Make merge key types consistent
model_data['season'] = model_data['season'].astype(int)
model_data['week'] = model_data['week'].astype(int)

ff_opp_pd['season'] = ff_opp_pd['season'].astype(int)
ff_opp_pd['week'] = ff_opp_pd['week'].astype(int)


# Remove opportunity columns if this cell has been run before
opp_cols_to_remove = [
    'opp_team',
    'opp_player_id',
    'my_expected_opportunity_points',
    'rush_yards_gained_exp',
    'rec_yards_gained_exp',
    'receptions_exp',
    'rush_touchdown_exp',
    'rec_touchdown_exp',
    'pass_yards_gained_exp',
    'pass_touchdown_exp'
]

model_data = model_data.drop(
    columns=[c for c in opp_cols_to_remove if c in model_data.columns]
)


# Prepare opportunity data
opp_merge = ff_opp_pd[[
    'season',
    'week',
    'posteam',
    'player_id',
    'my_expected_opportunity_points',
    'rush_yards_gained_exp',
    'rec_yards_gained_exp',
    'receptions_exp',
    'rush_touchdown_exp',
    'rec_touchdown_exp',
    'pass_yards_gained_exp',
    'pass_touchdown_exp'
]].copy()

opp_merge = opp_merge.rename(columns={
    'posteam': 'opp_team',
    'player_id': 'opp_player_id'
})


# Merge into master dataset
model_data = model_data.merge(
    opp_merge,
    left_on=[
        'season',
        'week',
        'recent_team',
        'player_id'
    ],
    right_on=[
        'season',
        'week',
        'opp_team',
        'opp_player_id'
    ],
    how='left'
)

print("Rows:", len(model_data))
print(
    "Rows with opportunity data:",
    model_data['my_expected_opportunity_points'].notna().sum()
)
print(
    "Rows missing opportunity data:",
    model_data['my_expected_opportunity_points'].isna().sum()
)

Rows: 21071
Rows with opportunity data: 21049
Rows missing opportunity data: 22


In [89]:
# Put everything in chronological order
model_data = model_data.sort_values(
    ['player_id', 'season', 'week']
).copy()

# Opportunity metrics we want to know entering each game
opp_cols = [
    'my_expected_opportunity_points',
    'rush_yards_gained_exp',
    'rec_yards_gained_exp',
    'receptions_exp',
    'rush_touchdown_exp',
    'rec_touchdown_exp',
    'pass_yards_gained_exp',
    'pass_touchdown_exp'
]

# Create PRIOR 3-game averages
for col in opp_cols:
    model_data[f'{col}_last3'] = (
        model_data
        .groupby(['player_id', 'season'])[col]
        .transform(
            lambda x: x.shift(1).rolling(
                3,
                min_periods=1
            ).mean()
        )
    )

print("Pregame expected-opportunity features created.")

Pregame expected-opportunity features created.


In [90]:
# Opportunity features that apply differently by position

opp_features = {

    'QB': [
        'my_expected_opportunity_points_last3',
        'pass_yards_gained_exp_last3',
        'pass_touchdown_exp_last3'
    ],

    'RB': [
        'my_expected_opportunity_points_last3',
        'rush_yards_gained_exp_last3',
        'rec_yards_gained_exp_last3',
        'receptions_exp_last3',
        'rush_touchdown_exp_last3',
        'rec_touchdown_exp_last3'
    ],

    'WR': [
        'my_expected_opportunity_points_last3',
        'rec_yards_gained_exp_last3',
        'receptions_exp_last3',
        'rec_touchdown_exp_last3'
    ],

    'TE': [
        'my_expected_opportunity_points_last3',
        'rec_yards_gained_exp_last3',
        'receptions_exp_last3',
        'rec_touchdown_exp_last3'
    ]
}


# Our current best feature sets

current_features = {

    'QB': [
        'my_fantasy_points_last3',
        'attempts_last3',
        'completions_last3',
        'passing_yards_last3',
        'passing_tds_last3',
        'interceptions_last3',
        'passing_air_yards_last3',
        'passing_epa_last3',
        'carries_last3',
        'rushing_yards_last3',
        'rushing_tds_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'is_home',
        'temp',
        'wind',
        'rest_days'
    ],

    'RB': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend',
        'on_injury_report',
        'questionable',
        'practice_full',
        'practice_limited',
        'practice_dnp'
    ],

    'WR': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend',
        'on_injury_report',
        'questionable',
        'practice_full',
        'practice_limited',
        'practice_dnp'
    ],

    'TE': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend',
        'on_injury_report',
        'questionable',
        'practice_full',
        'practice_limited',
        'practice_dnp'
    ]
}


results = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()
    test = pos_data[pos_data['season'] == 2024].copy()

    for label, cols in [
        ('Current model', current_features[pos]),
        ('+ Expected opportunity',
         current_features[pos] + opp_features[pos])
    ]:

        X_train = train[cols].copy()
        X_test = test[cols].copy()

        medians = X_train.median(numeric_only=True)

        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        m = RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )

        m.fit(
            X_train,
            train['my_fantasy_points']
        )

        preds = m.predict(X_test)

        mae = mean_absolute_error(
            test['my_fantasy_points'],
            preds
        )

        rank = spearmanr(
            preds,
            test['my_fantasy_points']
        ).statistic

        results.append({
            'Position': pos,
            'Model': label,
            'MAE': round(mae, 3),
            'Rank correlation': round(rank, 3)
        })

pd.DataFrame(results)

,Position,Model,MAE,Rank correlation
0,QB,Current model,6.337,0.495
1,QB,+ Expected opportunity,6.343,0.502
2,RB,Current model,4.671,0.602
3,RB,+ Expected opportunity,4.674,0.602
4,WR,Current model,4.425,0.542
5,WR,+ Expected opportunity,4.431,0.541
6,TE,Current model,3.342,0.505
7,TE,+ Expected opportunity,3.354,0.507


In [91]:
import itertools
import numpy as np

pair_results = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()
    test = pos_data[pos_data['season'] == 2024].copy()

    for label, cols in [
        ('Current model', current_features[pos]),
        ('+ Expected opportunity',
         current_features[pos] + opp_features[pos])
    ]:

        X_train = train[cols].copy()
        X_test = test[cols].copy()

        medians = X_train.median(numeric_only=True)

        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        m = RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )

        m.fit(
            X_train,
            train['my_fantasy_points']
        )

        test_eval = test[[
            'player_display_name',
            'week',
            'my_fantasy_points'
        ]].copy()

        test_eval['prediction'] = m.predict(X_test)

        correct = 0
        total = 0

        # Compare players only within the same week
        for week, group in test_eval.groupby('week'):

            rows = group.to_dict('records')

            for a, b in itertools.combinations(rows, 2):

                # Ignore exact ties
                if a['my_fantasy_points'] == b['my_fantasy_points']:
                    continue

                predicted_winner = (
                    'a' if a['prediction'] > b['prediction'] else 'b'
                )

                actual_winner = (
                    'a'
                    if a['my_fantasy_points'] > b['my_fantasy_points']
                    else 'b'
                )

                if predicted_winner == actual_winner:
                    correct += 1

                total += 1

        accuracy = correct / total

        pair_results.append({
            'Position': pos,
            'Model': label,
            'Pairwise accuracy': round(accuracy, 3),
            'Comparisons': total
        })

pd.DataFrame(pair_results)

,Position,Model,Pairwise accuracy,Comparisons
0,QB,Current model,0.672,9877
1,QB,+ Expected opportunity,0.678,9877
2,RB,Current model,0.715,42371
3,RB,+ Expected opportunity,0.715,42371
4,WR,Current model,0.690,105168
5,WR,+ Expected opportunity,0.690,105168
6,TE,Current model,0.681,26963
7,TE,+ Expected opportunity,0.681,26963


In [92]:
confidence_results = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()
    test = pos_data[pos_data['season'] == 2024].copy()

    # Use expected opportunity only for QB
    if pos == 'QB':
        cols = current_features[pos] + opp_features[pos]
    else:
        cols = current_features[pos]

    X_train = train[cols].copy()
    X_test = test[cols].copy()

    medians = X_train.median(numeric_only=True)

    X_train = X_train.fillna(medians)
    X_test = X_test.fillna(medians)

    m = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    m.fit(X_train, train['my_fantasy_points'])

    test_eval = test[[
        'player_display_name',
        'week',
        'my_fantasy_points'
    ]].copy()

    test_eval['prediction'] = m.predict(X_test)

    pair_rows = []

    for week, group in test_eval.groupby('week'):

        rows = group.to_dict('records')

        for a, b in itertools.combinations(rows, 2):

            if a['my_fantasy_points'] == b['my_fantasy_points']:
                continue

            pred_gap = abs(
                a['prediction'] - b['prediction']
            )

            predicted_a = (
                a['prediction'] > b['prediction']
            )

            actual_a = (
                a['my_fantasy_points'] >
                b['my_fantasy_points']
            )

            pair_rows.append({
                'gap': pred_gap,
                'correct': predicted_a == actual_a
            })

    pair_df = pd.DataFrame(pair_rows)

    # Confidence buckets
    bins = [0, 1, 2, 3, 5, 100]
    labels = [
        '<1',
        '1-2',
        '2-3',
        '3-5',
        '5+'
    ]

    pair_df['gap_bucket'] = pd.cut(
        pair_df['gap'],
        bins=bins,
        labels=labels,
        right=False
    )

    summary = (
        pair_df
        .groupby('gap_bucket', observed=True)
        .agg(
            accuracy=('correct', 'mean'),
            comparisons=('correct', 'size')
        )
        .reset_index()
    )

    summary['Position'] = pos

    confidence_results.append(summary)

confidence_table = pd.concat(
    confidence_results,
    ignore_index=True
)

confidence_table

,gap_bucket,accuracy,comparisons,Position
0,<1,0.539896,1341,QB
1,1-2,0.543846,1243,QB
2,2-3,0.607979,1153,QB
3,3-5,0.645017,2017,QB
4,5+,0.797963,4123,QB
5,<1,0.540446,6552,RB
6,1-2,0.568533,5377,RB
7,2-3,0.623500,4417,RB
8,3-5,0.696894,7245,RB
9,5+,0.847338,18780,RB


In [93]:
print("=== PROJECT CHECKPOINT ===")
print("model_data exists:", 'model_data' in globals())

if 'model_data' in globals():
    print("model_data shape:", model_data.shape)

print("Next step:")
print("Build the 2024 start/sit recommendation function with confidence scores.")

=== PROJECT CHECKPOINT ===
model_data exists: True
model_data shape: (21071, 125)
Next step:
Build the 2024 start/sit recommendation function with confidence scores.


In [94]:
# Train and store the final position models we want to use

final_models = {}
final_features = {}
final_medians = {}

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()

    # Expected opportunity helped QB ranking slightly,
    # so keep it for QB only.
    if pos == 'QB':
        cols = current_features[pos] + opp_features[pos]
    else:
        cols = current_features[pos]

    X_train = train[cols].copy()

    medians = X_train.median(numeric_only=True)

    X_train = X_train.fillna(medians)

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        train['my_fantasy_points']
    )

    final_models[pos] = model
    final_features[pos] = cols
    final_medians[pos] = medians

print("Final QB/RB/WR/TE models trained and stored.")

Final QB/RB/WR/TE models trained and stored.


In [119]:
def get_confidence(position, gap):

    position = position.upper()

    if position not in confidence_models:
        return None

    probability = confidence_models[position].predict_proba(
        np.array([[gap]])
    )[0, 1]

    return probability


def confidence_label(confidence):

    if confidence is None:
        return "Unknown"

    if confidence < 0.45:
        return "Very Low"
    elif confidence < 0.50:
        return "Low"
    elif confidence < 0.60:
        return "Moderate"
    elif confidence < 0.70:
        return "High"
    else:
        return "Very High"


def explain_model_drivers(selected, position):

    top = selected.iloc[0].copy()

    cols = final_features[position]
    model = final_models[position]
    medians = final_medians[position]

    # Create the top player's original feature row
    original = pd.DataFrame(
        [top[cols]]
    ).copy()

    original = original.fillna(medians)

    original_prediction = model.predict(
        original
    )[0]

    driver_rows = []

    friendly_names = {
        'my_fantasy_points_last3':
            'Recent fantasy production',

        'carries_last3':
            'Recent carries',

        'targets_last3':
            'Recent targets',

        'rushing_yards_last3':
            'Recent rushing yards',

        'receiving_yards_last3':
            'Recent receiving yards',

        'target_share_last3':
            'Recent target share',

        'offense_pct_last3':
            'Recent snap share',

        'offense_pct_last5':
            'Longer-term snap share',

        'snap_trend':
            'Recent snap trend',

        'team_spread':
            'Game spread',

        'total_line':
            'Game total',

        'defense_fp_allowed_last3':
            'Opponent matchup',

        'passing_yards_last3':
            'Recent passing yards',

        'passing_tds_last3':
            'Recent passing TDs',

        'attempts_last3':
            'Recent pass attempts',

        'completions_last3':
            'Recent completions',

        'passing_air_yards_last3':
            'Recent passing air yards',

        'passing_epa_last3':
            'Recent passing efficiency',

        'rushing_tds_last3':
            'Recent rushing TDs',

        'interceptions_last3':
            'Recent interceptions',

        'is_home':
            'Home-field status',

        'temp':
            'Temperature',

        'wind':
            'Wind',

        'rest_days':
            'Rest advantage',

        'on_injury_report':
            'Injury-report status',

        'questionable':
            'Questionable status',

        'practice_full':
            'Full-practice status',

        'practice_limited':
            'Limited-practice status',

        'practice_dnp':
            'Missed-practice status',

        'my_expected_opportunity_points_last3':
            'Recent expected opportunity',

        'pass_yards_gained_exp_last3':
            'Recent expected passing yards',

        'pass_touchdown_exp_last3':
            'Recent expected passing TDs'
    }

    # Test each feature by replacing it with
    # the training-set median and seeing how
    # much the projection changes
    for col in cols:

        if col not in medians.index:
            continue

        altered = original.copy()

        altered.loc[
            altered.index[0],
            col
        ] = medians[col]

        altered_prediction = model.predict(
            altered
        )[0]

        impact = (
            original_prediction
            - altered_prediction
        )

        driver_rows.append({
            'feature': col,
            'label': friendly_names.get(
                col,
                col
            ),
            'impact': impact
        })

    drivers = pd.DataFrame(
        driver_rows
    )

    if len(drivers) == 0:
        return drivers

    drivers = drivers.sort_values(
        'impact',
        ascending=False
    )

    # We only want features currently helping
    # the recommended player's projection
    drivers = drivers[
        drivers['impact'] > 0
    ]

    return drivers.head(5)


def build_caution_reasons(
    selected,
    position,
    gap,
    confidence
):

    cautions = []

    if confidence is None:
        return cautions

    # General confidence warning
    if confidence < 0.50:

        cautions.append(
            "This is a low-confidence decision based on "
            "historical 3-player start/sit results."
        )

    # Projection-gap warning
    if gap is not None and gap < 2:

        cautions.append(
            "The top two players are separated by less "
            "than 2 projected points."
        )

    elif gap is not None and gap < 5:

        cautions.append(
            "The projection gap is meaningful, but not "
            "large enough to make this a strong call."
        )

    # Compare top two players
    if len(selected) >= 2:

        top = selected.iloc[0]
        second = selected.iloc[1]

        # Similar recent fantasy production
        if (
            'my_fantasy_points_last3'
            in selected.columns
            and pd.notna(
                top['my_fantasy_points_last3']
            )
            and pd.notna(
                second['my_fantasy_points_last3']
            )
        ):

            recent_gap = abs(
                top['my_fantasy_points_last3']
                - second['my_fantasy_points_last3']
            )

            if recent_gap < 2:

                cautions.append(
                    f"{second['player_display_name']} has "
                    "very similar recent fantasy production."
                )

        # Position-specific workload comparison
        workload_col = None

        if position == 'RB':
            workload_col = 'carries_last3'

        elif position in ['WR', 'TE']:
            workload_col = 'targets_last3'

        elif position == 'QB':
            workload_col = 'attempts_last3'

        if (
            workload_col is not None
            and workload_col in selected.columns
            and pd.notna(
                top[workload_col]
            )
            and pd.notna(
                second[workload_col]
            )
        ):

            workload_gap = abs(
                top[workload_col]
                - second[workload_col]
            )

            if workload_gap < 2:

                cautions.append(
                    f"{second['player_display_name']} has "
                    "a very similar recent workload."
                )

    return cautions[:3]


def start_sit_2024(
    week,
    position,
    players
):

    position = position.upper()

    if position not in [
        'QB',
        'RB',
        'WR',
        'TE'
    ]:

        print(
            "Position must be QB, RB, WR, or TE."
        )

        return

    # -----------------------------------
    # GET PLAYERS FOR THIS WEEK
    # -----------------------------------

    week_data = model_data[
        (model_data['season'] == 2024) &
        (model_data['week'] == week) &
        (model_data['position'] == position)
    ].copy()

    selected = week_data[
        week_data[
            'player_display_name'
        ].isin(players)
    ].copy()

    if len(selected) == 0:

        print(
            "No matching players found."
        )

        return

    # Warn if a supplied player wasn't found
    found = set(
        selected[
            'player_display_name'
        ]
    )

    missing = [
        p
        for p in players
        if p not in found
    ]

    if missing:

        print(
            "Could not find:",
            missing
        )

    # -----------------------------------
    # CREATE MODEL PROJECTIONS
    # -----------------------------------

    cols = final_features[
        position
    ]

    X = selected[
        cols
    ].copy()

    X = X.fillna(
        final_medians[
            position
        ]
    )

    selected[
        'model_projection'
    ] = (
        final_models[
            position
        ].predict(X)
    )

    selected = (
        selected
        .sort_values(
            'model_projection',
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    selected[
        'rank'
    ] = (
        selected.index + 1
    )

    # -----------------------------------
    # CONFIDENCE
    # -----------------------------------

    if len(selected) >= 2:

        gap = (
            selected.loc[
                0,
                'model_projection'
            ]
            - selected.loc[
                1,
                'model_projection'
            ]
        )

        confidence = (
            get_confidence(
                position,
                gap
            )
        )

    else:

        gap = None
        confidence = None

    label = confidence_label(
        confidence
    )

    # -----------------------------------
    # MODEL-AWARE EXPLANATION
    # -----------------------------------

    driver_table = (
        explain_model_drivers(
            selected,
            position
        )
    )

    reasons = (
        driver_table.head(3)
    )

    cautions = (
        build_caution_reasons(
            selected,
            position,
            gap,
            confidence
        )
    )

    # -----------------------------------
    # HISTORICAL RESULT
    # -----------------------------------

    actual_winner_idx = (
        selected[
            'my_fantasy_points'
        ].idxmax()
    )

    actual_winner = (
        selected.loc[
            actual_winner_idx,
            'player_display_name'
        ]
    )

    recommendation = (
        selected.loc[
            0,
            'player_display_name'
        ]
    )

    correct_pick = (
        recommendation
        == actual_winner
    )

    # -----------------------------------
    # DISPLAY
    # -----------------------------------

    print()
    print(
        "=" * 50
    )

    print(
        f"2024 WEEK {week} — "
        f"{position} START/SIT"
    )

    print(
        "=" * 50
    )

    print()
    print(
        f"START: {recommendation}"
    )

    print()
    print(
        "MODEL RANKINGS"
    )

    for i, row in selected.iterrows():

        print(
            f"{i + 1}. "
            f"{row['player_display_name']} "
            f"({row['recent_team']} vs "
            f"{row['opponent_team']}) "
            f"— "
            f"{row['model_projection']:.2f} pts"
        )

    # -----------------------------------
    # CONFIDENCE DISPLAY
    # -----------------------------------

    if confidence is not None:

        print()

        print(
            f"Projection gap over #2: "
            f"{gap:.2f} points"
        )

        print(
            f"Historical confidence: "
            f"{confidence:.1%} "
            f"({label})"
        )

    # -----------------------------------
    # WHY THE MODEL LIKES THE PICK
    # -----------------------------------

    print()
    print(
        "WHY THE MODEL LIKES THIS PICK"
    )

    if len(reasons) > 0:

        for _, driver in reasons.iterrows():

            print(
                f"• {driver['label']} "
                f"is a strong positive driver "
                f"of the model's projection"
            )

    else:

        print(
            "• No single feature clearly "
            "drives the recommendation."
        )

    # -----------------------------------
    # REASONS FOR CAUTION
    # -----------------------------------

    if cautions:

        print()
        print(
            "REASONS FOR CAUTION"
        )

        for caution in cautions:

            print(
                f"• {caution}"
            )

    # -----------------------------------
    # HISTORICAL RESULT DISPLAY
    # -----------------------------------

    print()
    print(
        "HISTORICAL RESULT"
    )

    print(
        f"Actual top scorer: "
        f"{actual_winner}"
    )

    if correct_pick:

        print(
            "Model result: CORRECT"
        )

    else:

        print(
            "Model result: MISSED"
        )

    print()

    # -----------------------------------
    # RETURN TABLE
    # -----------------------------------

    result = selected[[
        'rank',
        'player_display_name',
        'recent_team',
        'opponent_team',
        'model_projection',
        'my_fantasy_points'
    ]].copy()

    result[
        'model_projection'
    ] = (
        result[
            'model_projection'
        ].round(2)
    )

    result[
        'my_fantasy_points'
    ] = (
        result[
            'my_fantasy_points'
        ].round(2)
    )

    return result

In [120]:
start_sit_2024(
    week=10,
    position='RB',
    players=[
        'Saquon Barkley',
        'Bijan Robinson',
        'Breece Hall'
    ]
)


2024 WEEK 10 — RB START/SIT

START: Saquon Barkley

MODEL RANKINGS
1. Saquon Barkley (PHI vs DAL) — 20.45 pts
2. Bijan Robinson (ATL vs NO) — 16.61 pts
3. Breece Hall (NYJ vs ARI) — 13.81 pts

Projection gap over #2: 3.84 points
Historical confidence: 45.6% (Low)

WHY THE MODEL LIKES THIS PICK
• Recent snap share is a strong positive driver of the model's projection
• Recent carries is a strong positive driver of the model's projection
• Recent fantasy production is a strong positive driver of the model's projection

REASONS FOR CAUTION
• This is a low-confidence decision based on historical 3-player start/sit results.
• The projection gap is meaningful, but not large enough to make this a strong call.

HISTORICAL RESULT
Actual top scorer: Bijan Robinson
Model result: MISSED



,rank,player_display_name,recent_team,opponent_team,model_projection,my_fantasy_points
0,1,Saquon Barkley,PHI,DAL,20.45,8.3
1,2,Bijan Robinson,ATL,NO,16.61,27.9
2,3,Breece Hall,NYJ,ARI,13.81,10.3


In [116]:
def explain_model_drivers(selected, position):

    top = selected.iloc[0].copy()

    cols = final_features[position]
    model = final_models[position]
    medians = final_medians[position]

    # Original feature row
    original = pd.DataFrame(
        [top[cols]]
    ).copy()

    original = original.fillna(medians)

    original_prediction = model.predict(
        original
    )[0]

    driver_rows = []

    friendly_names = {
        'my_fantasy_points_last3':
            'Recent fantasy production',

        'carries_last3':
            'Recent carries',

        'targets_last3':
            'Recent targets',

        'rushing_yards_last3':
            'Recent rushing yards',

        'receiving_yards_last3':
            'Recent receiving yards',

        'target_share_last3':
            'Recent target share',

        'offense_pct_last3':
            'Recent snap share',

        'offense_pct_last5':
            'Longer-term snap share',

        'team_spread':
            'Game spread',

        'total_line':
            'Game total',

        'defense_fp_allowed_last3':
            'Opponent matchup',

        'passing_yards_last3':
            'Recent passing yards',

        'passing_tds_last3':
            'Recent passing TDs',

        'attempts_last3':
            'Recent pass attempts',

        'completions_last3':
            'Recent completions',

        'rushing_tds_last3':
            'Recent rushing TDs'
    }

    for col in cols:

        if col not in medians.index:
            continue

        altered = original.copy()

        # Replace this one feature with league/training median
        altered.loc[
            altered.index[0],
            col
        ] = medians[col]

        altered_prediction = model.predict(
            altered
        )[0]

        impact = (
            original_prediction -
            altered_prediction
        )

        driver_rows.append({
            'feature': col,
            'label': friendly_names.get(
                col,
                col
            ),
            'impact': impact
        })

    drivers = pd.DataFrame(
        driver_rows
    )

    drivers = drivers.sort_values(
        'impact',
        ascending=False
    )

    # Only positive drivers
    drivers = drivers[
        drivers['impact'] > 0
    ]

    return drivers.head(5)

In [117]:
# Recreate the Saquon/Bijan/Breece prediction
week_data = model_data[
    (model_data['season'] == 2024) &
    (model_data['week'] == 10) &
    (model_data['position'] == 'RB')
].copy()

selected_test = week_data[
    week_data['player_display_name'].isin([
        'Saquon Barkley',
        'Bijan Robinson',
        'Breece Hall'
    ])
].copy()

X_test = selected_test[
    final_features['RB']
].copy()

X_test = X_test.fillna(
    final_medians['RB']
)

selected_test['model_projection'] = (
    final_models['RB'].predict(X_test)
)

selected_test = selected_test.sort_values(
    'model_projection',
    ascending=False
).reset_index(drop=True)

explain_model_drivers(
    selected_test,
    'RB'
)

,feature,label,impact
9,offense_pct_last3,Recent snap share,7.529060
1,carries_last3,Recent carries,4.609844
0,my_fantasy_points_last3,Recent fantasy production,4.115635
7,team_spread,Game spread,3.853416
3,rushing_yards_last3,Recent rushing yards,2.633428


In [115]:
start_sit_2024(
    week=10,
    position='RB',
    players=[
        'Saquon Barkley',
        'Bijan Robinson',
        'Breece Hall'
    ]
)


2024 WEEK 10 — RB START/SIT

START: Saquon Barkley

MODEL RANKINGS
1. Saquon Barkley (PHI vs DAL) — 20.45 pts
2. Bijan Robinson (ATL vs NO) — 16.61 pts
3. Breece Hall (NYJ vs ARI) — 13.81 pts

Projection gap over #2: 3.84 points
Historical confidence: 45.6% (Low)

WHY THE MODEL LIKES THIS PICK
• Recent rushing yards: Saquon Barkley 147.7 vs others 74.0
• Game spread: Saquon Barkley -7.5 vs others -2.8
• Recent carries: Saquon Barkley 22.0 vs others 16.0

REASONS FOR CAUTION
• This is a low-confidence decision based on historical 3-player start/sit results.
• The projection gap is meaningful, but not large enough to make this a strong call.

HISTORICAL RESULT
Actual top scorer: Bijan Robinson
Model result: MISSED



,rank,player_display_name,recent_team,opponent_team,model_projection,my_fantasy_points
0,1,Saquon Barkley,PHI,DAL,20.45,8.3
1,2,Bijan Robinson,ATL,NO,16.61,27.9
2,3,Breece Hall,NYJ,ARI,13.81,10.3


In [97]:
# Test RB confidence only among fantasy-relevant RBs

pos = 'RB'

pos_data = model_data[
    (model_data['position'] == pos) &
    (model_data['my_fantasy_points_last3'].notna())
].copy()

train = pos_data[pos_data['season'] <= 2023].copy()
test = pos_data[pos_data['season'] == 2024].copy()

cols = final_features[pos]

X_train = train[cols].copy()
X_test = test[cols].copy()

medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    train['my_fantasy_points']
)

test['prediction'] = model.predict(X_test)

realistic_pairs = []

for week, group in test.groupby('week'):

    # Only consider the 30 RBs the model viewed
    # as most fantasy-relevant entering the week
    group = group.nlargest(
        30,
        'prediction'
    )

    rows = group.to_dict('records')

    for a, b in itertools.combinations(rows, 2):

        if a['my_fantasy_points'] == b['my_fantasy_points']:
            continue

        gap = abs(
            a['prediction'] - b['prediction']
        )

        predicted_a = (
            a['prediction'] > b['prediction']
        )

        actual_a = (
            a['my_fantasy_points'] >
            b['my_fantasy_points']
        )

        realistic_pairs.append({
            'gap': gap,
            'correct': predicted_a == actual_a
        })

realistic_pairs = pd.DataFrame(realistic_pairs)

bins = [0, 1, 2, 3, 5, 100]
labels = [
    '<1',
    '1-2',
    '2-3',
    '3-5',
    '5+'
]

realistic_pairs['gap_bucket'] = pd.cut(
    realistic_pairs['gap'],
    bins=bins,
    labels=labels,
    right=False
)

realistic_summary = (
    realistic_pairs
    .groupby('gap_bucket', observed=True)
    .agg(
        accuracy=('correct', 'mean'),
        comparisons=('correct', 'size')
    )
    .reset_index()
)

print("RB — Top 30 fantasy-relevant players only")
realistic_summary

RB — Top 30 fantasy-relevant players only


,gap_bucket,accuracy,comparisons
0,<1,0.538680,1758
1,1-2,0.534674,1442
2,2-3,0.554531,1302
3,3-5,0.628382,1663
4,5+,0.675497,1208


In [98]:
realistic_pool_sizes = {
    'QB': 18,
    'RB': 30,
    'WR': 40,
    'TE': 18
}

all_realistic_confidence = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()
    test = pos_data[pos_data['season'] == 2024].copy()

    cols = final_features[pos]

    X_train = train[cols].copy()
    X_test = test[cols].copy()

    medians = X_train.median(numeric_only=True)

    X_train = X_train.fillna(medians)
    X_test = X_test.fillna(medians)

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        train['my_fantasy_points']
    )

    test['prediction'] = model.predict(X_test)

    pair_rows = []

    for week, group in test.groupby('week'):

        group = group.nlargest(
            realistic_pool_sizes[pos],
            'prediction'
        )

        rows = group.to_dict('records')

        for a, b in itertools.combinations(rows, 2):

            if a['my_fantasy_points'] == b['my_fantasy_points']:
                continue

            gap = abs(
                a['prediction'] - b['prediction']
            )

            predicted_a = (
                a['prediction'] > b['prediction']
            )

            actual_a = (
                a['my_fantasy_points'] >
                b['my_fantasy_points']
            )

            pair_rows.append({
                'gap': gap,
                'correct': predicted_a == actual_a
            })

    pair_df = pd.DataFrame(pair_rows)

    bins = [0, 1, 2, 3, 5, 100]
    labels = ['<1', '1-2', '2-3', '3-5', '5+']

    pair_df['gap_bucket'] = pd.cut(
        pair_df['gap'],
        bins=bins,
        labels=labels,
        right=False
    )

    summary = (
        pair_df
        .groupby('gap_bucket', observed=True)
        .agg(
            accuracy=('correct', 'mean'),
            comparisons=('correct', 'size')
        )
        .reset_index()
    )

    summary['Position'] = pos

    all_realistic_confidence.append(summary)

realistic_confidence_table = pd.concat(
    all_realistic_confidence,
    ignore_index=True
)

realistic_confidence_table

,gap_bucket,accuracy,comparisons,Position
0,<1,0.532934,668,QB
1,1-2,0.537071,553,QB
2,2-3,0.590698,430,QB
3,3-5,0.623917,577,QB
4,5+,0.666667,372,QB
5,<1,0.538680,1758,RB
6,1-2,0.534674,1442,RB
7,2-3,0.554531,1302,RB
8,3-5,0.628382,1663,RB
9,5+,0.675497,1208,RB


In [99]:
three_player_results = []

realistic_pool_sizes = {
    'QB': 18,
    'RB': 30,
    'WR': 40,
    'TE': 18
}

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()
    test = pos_data[pos_data['season'] == 2024].copy()

    cols = final_features[pos]

    X_train = train[cols].copy()
    X_test = test[cols].copy()

    medians = X_train.median(numeric_only=True)

    X_train = X_train.fillna(medians)
    X_test = X_test.fillna(medians)

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        train['my_fantasy_points']
    )

    test['prediction'] = model.predict(X_test)

    decision_rows = []

    for week, group in test.groupby('week'):

        # Restrict to realistic fantasy player pool
        group = group.nlargest(
            realistic_pool_sizes[pos],
            'prediction'
        )

        rows = group.to_dict('records')

        # Every possible group of 3 players
        for trio in itertools.combinations(rows, 3):

            trio = sorted(
                trio,
                key=lambda x: x['prediction'],
                reverse=True
            )

            model_pick = trio[0]

            # Difference between model's #1 and #2
            gap = (
                trio[0]['prediction'] -
                trio[1]['prediction']
            )

            actual_scores = [
                p['my_fantasy_points']
                for p in trio
            ]

            # Ignore exact ties for highest actual score
            max_actual = max(actual_scores)

            winners = [
                p for p in trio
                if p['my_fantasy_points'] == max_actual
            ]

            if len(winners) != 1:
                continue

            correct = (
                model_pick['player_display_name']
                ==
                winners[0]['player_display_name']
            )

            decision_rows.append({
                'gap': gap,
                'correct': correct
            })

    decisions = pd.DataFrame(decision_rows)

    bins = [0, 1, 2, 3, 5, 100]

    labels = [
        '<1',
        '1-2',
        '2-3',
        '3-5',
        '5+'
    ]

    decisions['gap_bucket'] = pd.cut(
        decisions['gap'],
        bins=bins,
        labels=labels,
        right=False
    )

    summary = (
        decisions
        .groupby(
            'gap_bucket',
            observed=True
        )
        .agg(
            accuracy=('correct', 'mean'),
            decisions=('correct', 'size')
        )
        .reset_index()
    )

    summary['Position'] = pos

    three_player_results.append(summary)

three_player_confidence = pd.concat(
    three_player_results,
    ignore_index=True
)

three_player_confidence

,gap_bucket,accuracy,decisions,Position
0,<1,0.407885,3957,QB
1,1-2,0.403846,3016,QB
2,2-3,0.453840,2578,QB
3,3-5,0.515382,2893,QB
4,5+,0.503516,1422,QB
5,<1,0.382246,21235,RB
6,1-2,0.396070,15419,RB
7,2-3,0.424828,11906,RB
8,3-5,0.491720,13044,RB
9,5+,0.515778,7162,RB


In [100]:
# Build calibration and validation datasets

calibration_models = {}
calibration_features = {}
calibration_medians = {}

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2022].copy()
    calibration = pos_data[pos_data['season'] == 2023].copy()
    validation = pos_data[pos_data['season'] == 2024].copy()

    cols = final_features[pos]

    X_train = train[cols].copy()
    medians = X_train.median(numeric_only=True)
    X_train = X_train.fillna(medians)

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        train['my_fantasy_points']
    )

    calibration_models[pos] = model
    calibration_features[pos] = cols
    calibration_medians[pos] = medians

    print(
        pos,
        "| train:", len(train),
        "| calibration:", len(calibration),
        "| validation:", len(validation)
    )

QB | train: 1149 | calibration: 582 | validation: 586
RB | train: 2451 | calibration: 1209 | validation: 1208
WR | train: 3884 | calibration: 1999 | validation: 1905
TE | train: 1896 | calibration: 959 | validation: 969


In [101]:
calibration_confidence_results = []

realistic_pool_sizes = {
    'QB': 18,
    'RB': 30,
    'WR': 40,
    'TE': 18
}

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    calibration = pos_data[
        pos_data['season'] == 2023
    ].copy()

    cols = calibration_features[pos]

    X_cal = calibration[cols].copy()
    X_cal = X_cal.fillna(calibration_medians[pos])

    calibration['prediction'] = (
        calibration_models[pos].predict(X_cal)
    )

    decision_rows = []

    for week, group in calibration.groupby('week'):

        group = group.nlargest(
            realistic_pool_sizes[pos],
            'prediction'
        )

        rows = group.to_dict('records')

        for trio in itertools.combinations(rows, 3):

            trio = sorted(
                trio,
                key=lambda x: x['prediction'],
                reverse=True
            )

            model_pick = trio[0]

            gap = (
                trio[0]['prediction'] -
                trio[1]['prediction']
            )

            actual_scores = [
                p['my_fantasy_points']
                for p in trio
            ]

            max_actual = max(actual_scores)

            winners = [
                p for p in trio
                if p['my_fantasy_points'] == max_actual
            ]

            if len(winners) != 1:
                continue

            correct = (
                model_pick['player_display_name']
                ==
                winners[0]['player_display_name']
            )

            decision_rows.append({
                'gap': gap,
                'correct': correct
            })

    decisions = pd.DataFrame(decision_rows)

    bins = [0, 1, 2, 3, 5, 100]
    labels = ['<1', '1-2', '2-3', '3-5', '5+']

    decisions['gap_bucket'] = pd.cut(
        decisions['gap'],
        bins=bins,
        labels=labels,
        right=False
    )

    summary = (
        decisions
        .groupby('gap_bucket', observed=True)
        .agg(
            accuracy=('correct', 'mean'),
            decisions=('correct', 'size')
        )
        .reset_index()
    )

    summary['Position'] = pos

    calibration_confidence_results.append(summary)

calibration_confidence_table = pd.concat(
    calibration_confidence_results,
    ignore_index=True
)

calibration_confidence_table

,gap_bucket,accuracy,decisions,Position
0,<1,0.376883,3850,QB
1,1-2,0.376074,3143,QB
2,2-3,0.399047,2308,QB
3,3-5,0.423203,2741,QB
4,5+,0.501379,1813,QB
5,<1,0.387909,19850,RB
6,1-2,0.419224,16979,RB
7,2-3,0.437240,12524,RB
8,3-5,0.467154,13944,RB
9,5+,0.575763,5438,RB


In [102]:
validation_results = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    validation = pos_data[
        pos_data['season'] == 2024
    ].copy()

    cols = calibration_features[pos]

    X_val = validation[cols].copy()
    X_val = X_val.fillna(calibration_medians[pos])

    validation['prediction'] = (
        calibration_models[pos].predict(X_val)
    )

    decision_rows = []

    for week, group in validation.groupby('week'):

        group = group.nlargest(
            realistic_pool_sizes[pos],
            'prediction'
        )

        rows = group.to_dict('records')

        for trio in itertools.combinations(rows, 3):

            trio = sorted(
                trio,
                key=lambda x: x['prediction'],
                reverse=True
            )

            model_pick = trio[0]

            gap = (
                trio[0]['prediction'] -
                trio[1]['prediction']
            )

            actual_scores = [
                p['my_fantasy_points']
                for p in trio
            ]

            max_actual = max(actual_scores)

            winners = [
                p for p in trio
                if p['my_fantasy_points'] == max_actual
            ]

            if len(winners) != 1:
                continue

            correct = (
                model_pick['player_display_name']
                ==
                winners[0]['player_display_name']
            )

            decision_rows.append({
                'gap': gap,
                'correct': correct
            })

    decisions = pd.DataFrame(decision_rows)

    bins = [0, 1, 2, 3, 5, 100]
    labels = ['<1', '1-2', '2-3', '3-5', '5+']

    decisions['gap_bucket'] = pd.cut(
        decisions['gap'],
        bins=bins,
        labels=labels,
        right=False
    )

    summary = (
        decisions
        .groupby('gap_bucket', observed=True)
        .agg(
            validation_accuracy=('correct', 'mean'),
            validation_decisions=('correct', 'size')
        )
        .reset_index()
    )

    summary['Position'] = pos

    validation_results.append(summary)

validation_confidence_table = pd.concat(
    validation_results,
    ignore_index=True
)

# Join 2023 learned confidence with 2024 actual accuracy
comparison = calibration_confidence_table.merge(
    validation_confidence_table,
    on=['Position', 'gap_bucket'],
    how='left'
)

comparison = comparison.rename(columns={
    'accuracy': '2023_calibrated_confidence',
    'decisions': '2023_decisions'
})

comparison['difference'] = (
    comparison['validation_accuracy']
    - comparison['2023_calibrated_confidence']
)

comparison

,gap_bucket,2023_calibrated_confidence,2023_decisions,Position,validation_accuracy,validation_decisions,difference
0,<1,0.376883,3850,QB,0.369929,3525,-0.006954
1,1-2,0.376074,3143,QB,0.414939,2798,0.038865
2,2-3,0.399047,2308,QB,0.482744,2318,0.083697
3,3-5,0.423203,2741,QB,0.428154,3083,0.004951
4,5+,0.501379,1813,QB,0.480617,2141,-0.020762
5,<1,0.387909,19850,RB,0.391375,20985,0.003465
6,1-2,0.419224,16979,RB,0.368505,16898,-0.050719
7,2-3,0.437240,12524,RB,0.400229,12218,-0.037011
8,3-5,0.467154,13944,RB,0.456588,13706,-0.010566
9,5+,0.575763,5438,RB,0.529352,4940,-0.046411


In [103]:
walkforward_predictions = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    cols = final_features[pos]

    # Predict each season using ONLY previous seasons
    for test_year in [2022, 2023, 2024]:

        train = pos_data[
            pos_data['season'] < test_year
        ].copy()

        test = pos_data[
            pos_data['season'] == test_year
        ].copy()

        if len(train) == 0 or len(test) == 0:
            continue

        X_train = train[cols].copy()
        X_test = test[cols].copy()

        medians = X_train.median(numeric_only=True)

        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            train['my_fantasy_points']
        )

        test['prediction'] = model.predict(X_test)
        test['Position'] = pos

        walkforward_predictions.append(
            test[[
                'season',
                'week',
                'player_display_name',
                'recent_team',
                'opponent_team',
                'Position',
                'prediction',
                'my_fantasy_points'
            ]]
        )

walkforward_df = pd.concat(
    walkforward_predictions,
    ignore_index=True
)

print("Walk-forward predictions created.")
print("Rows:", len(walkforward_df))

print(
    walkforward_df
    .groupby(['Position', 'season'])
    .size()
)

Walk-forward predictions created.
Rows: 14094
Position  season
QB        2022       564
          2023       582
          2024       586
RB        2022      1236
          2023      1209
          2024      1208
TE        2022       965
          2023       959
          2024       969
WR        2022      1912
          2023      1999
          2024      1905
dtype: int64


In [104]:
walkforward_decisions = []

realistic_pool_sizes = {
    'QB': 18,
    'RB': 30,
    'WR': 40,
    'TE': 18
}

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = walkforward_df[
        walkforward_df['Position'] == pos
    ].copy()

    for (season, week), group in pos_data.groupby(['season', 'week']):

        # Limit to realistic fantasy-relevant pool
        group = group.nlargest(
            realistic_pool_sizes[pos],
            'prediction'
        )

        rows = group.to_dict('records')

        # Every possible 3-player decision
        for trio in itertools.combinations(rows, 3):

            trio = sorted(
                trio,
                key=lambda x: x['prediction'],
                reverse=True
            )

            model_pick = trio[0]

            gap = (
                trio[0]['prediction'] -
                trio[1]['prediction']
            )

            actual_scores = [
                p['my_fantasy_points']
                for p in trio
            ]

            max_actual = max(actual_scores)

            winners = [
                p for p in trio
                if p['my_fantasy_points'] == max_actual
            ]

            # Ignore exact ties for highest scorer
            if len(winners) != 1:
                continue

            correct = (
                model_pick['player_display_name']
                ==
                winners[0]['player_display_name']
            )

            walkforward_decisions.append({
                'Position': pos,
                'season': season,
                'week': week,
                'gap': gap,
                'correct': correct
            })

walkforward_decisions = pd.DataFrame(
    walkforward_decisions
)

print("Walk-forward 3-player decisions created.")
print("Total decisions:", len(walkforward_decisions))

walkforward_decisions.groupby('Position').agg(
    decisions=('correct', 'size'),
    overall_accuracy=('correct', 'mean')
)

Walk-forward 3-player decisions created.
Total decisions: 790522


,decisions,overall_accuracy
Position,,
QB,41566,0.433672
RB,206015,0.424474
TE,41379,0.433432
WR,501562,0.429624


In [105]:
from sklearn.linear_model import LogisticRegression
import numpy as np

confidence_models = {}
confidence_summary = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    df = walkforward_decisions[
        walkforward_decisions['Position'] == pos
    ].copy()

    X = df[['gap']].values
    y = df['correct'].astype(int).values

    clf = LogisticRegression()
    clf.fit(X, y)

    confidence_models[pos] = clf

    # Show estimated confidence at useful projection gaps
    for gap in [0, 1, 2, 3, 4, 5, 7.5, 10]:

        prob = clf.predict_proba(
            np.array([[gap]])
        )[0, 1]

        confidence_summary.append({
            'Position': pos,
            'Projection gap': gap,
            'Estimated confidence': round(prob, 3)
        })

confidence_curve_table = pd.DataFrame(
    confidence_summary
)

confidence_curve_table

,Position,Projection gap,Estimated confidence
0,QB,0.0,0.378
1,QB,1.0,0.401
2,QB,2.0,0.425
3,QB,3.0,0.449
4,QB,4.0,0.473
5,QB,5.0,0.498
6,QB,7.5,0.559
7,QB,10.0,0.618
8,RB,0.0,0.375
9,RB,1.0,0.395


In [106]:
def get_confidence(position, gap):

    position = position.upper()

    if position not in confidence_models:
        return None

    probability = confidence_models[position].predict_proba(
        np.array([[gap]])
    )[0, 1]

    return probability

In [107]:
# Confidence is based on top player vs second-best player
if len(selected) >= 2:

    gap = (
        selected.loc[0, 'model_projection'] -
        selected.loc[1, 'model_projection']
    )

    confidence = get_confidence(position, gap)

else:
    gap = None
    confidence = None

NameError: name 'selected' is not defined

In [121]:
print("=== PROJECT CHECKPOINT ===")
print("Historical start/sit engine is working.")
print("Current next step:")
print("Combine model-aware drivers with actual values in the WHY section.")
print("After that: build current-season/live weekly data pipeline.")

=== PROJECT CHECKPOINT ===
Historical start/sit engine is working.
Current next step:
Combine model-aware drivers with actual values in the WHY section.
After that: build current-season/live weekly data pipeline.
